# Capítulo 16: Deep Learning

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 19 de Grus (2019).

> Um pouco de saber é coisa perigosa; beba fundo, ou não prove da fonte Piéria.
>
> — Alexander Pope

**Este capítulo não ensina um modelo novo.** Ele pega a rede do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) — uma lista de listas de listas de `float`, com a retropropagação escrita à mão para exatamente duas camadas — e a reescreve como uma **biblioteca**: um punhado de classes chamadas `Layer`, `Linear`, `Sequential`, `Loss` e `Optimizer`, que se encaixam umas nas outras.

Esses nomes não são invenção do livro-texto. São, letra por letra, os nomes que o PyTorch e o TensorFlow expõem. Quando alguém escreve

```python
model = Sequential([Linear(784, 30), Tanh(), Linear(30, 10)])
```

num framework de verdade, o que está do outro lado da chamada é o que este capítulo constrói do zero, em Python puro, em oito seções. É o capítulo em que a caixa-preta mais usada da área é aberta — e o que se encontra dentro dela não é um algoritmo novo, é uma **decisão de arquitetura**: separar o que cada camada calcula de como o gradiente atravessa essa camada, e deixar que as peças se componham.

O termo *deep learning* originalmente designava redes com mais de uma camada escondida. Hoje ele cobre uma variedade enorme de arquiteturas, incluindo as redes "simples" do capítulo anterior. O que muda aqui não é a profundidade em si; é que a profundidade passa a **custar nada para escrever**. A `sqerror_gradients` da [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) só funcionava para duas camadas, e cada topologia nova exigiria uma função nova. Depois deste capítulo, uma rede de quatro camadas é uma lista de quatro elementos.

> **❗ Importante — O preço aparece medido, e é conteúdo**
>
> Este é o capítulo mais caro do livro para renderizar, e isso não fica escondido.
>
> A [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) treina uma rede sobre imagens de dígitos manuscritos. Cada imagem custa **cerca de 6 milissegundos** na nossa biblioteca de listas — o que dá **cerca de seis minutos** para uma única passada pelas 60.000 imagens de treino. O corte que fazemos (10.000 imagens, três passadas) está escrito no texto, com os números, em vez de disfarçado.
>
> Isso não é uma desculpa pelo código lento. É a tese do livro ficando tangível: a lentidão é o preço da transparência, e aqui esse preço tem unidade e valor. Um aluno que passa por este capítulo entende, de uma vez e sem metáfora, por que existem GPUs.

Ao final deste capítulo, você será capaz de:

- Representar dados de qualquer dimensão como tensores e escrever funções recursivas que operam sobre eles
- Definir o contrato de uma camada — `forward`, `backward`, `params`, `grads` — e explicar por que ele é suficiente para treinar qualquer composição de camadas
- Implementar a camada linear e conferir os gradientes dela por diferenças finitas
- Compor redes de profundidade arbitrária e trocar ativação, perda e otimizador sem reescrever o laço de treino
- Explicar por que a softmax com entropia cruzada substituiu as sigmoides independentes com erro quadrático, e medir a diferença
- Aplicar dropout e explicar por que um modelo com dropout se comporta de forma diferente no treino e na avaliação
- Ler um formato binário de dados sem biblioteca e treinar um classificador de dígitos com a biblioteca que você escreveu
- Estimar o custo de treino de uma rede e dizer qual corte de escopo cabe no orçamento disponível

## Seções

| Seção | Tópico |
|---|---|
| [16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-o-tensor.html) | O Tensor |
| [16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-a-abstracao-de-camada.html) | A Abstração de Camada |
| [16.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/03-a-camada-linear.html) | A Camada Linear |
| [16.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/04-redes-como-sequencia-de-camadas.html) | Redes como Sequência de Camadas |
| [16.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/05-perda-e-otimizacao.html) | Perda e Otimização |
| [16.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/06-outras-funcoes-de-ativacao.html) | Outras Funções de Ativação |
| [16.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/07-softmax-e-dropout.html) | Softmax, Entropia Cruzada e Dropout |
| [16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) | Exemplo: MNIST |

## O Tensor

> **📌 Nota**
>
> Esta seção corresponde a *The Tensor*, do capítulo 19 de Grus (2019).

Até aqui o livro trabalhou com duas estruturas: **vetores**, que são arranjos de uma dimensão, e **matrizes**, que são de duas. O [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html) construiu as duas e as operações entre elas. Redes neurais mais complicadas precisam de mais: um lote de imagens coloridas, por exemplo, é um arranjo de quatro dimensões — imagem, linha, coluna, canal de cor.

Essa é a justificativa do livro-texto, e vale dizer desde já que ela não é a nossa: **as redes deste capítulo nunca passam de duas dimensões.** As imagens do MNIST da [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) chegam achatadas em 784 números, e todo tensor que entra ou sai de uma camada aqui é um vetor ou uma matriz. Arranjos de três dimensões aparecem aqui e ali — este capítulo mesmo constrói alguns, e a [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html) lê uma imagem colorida como altura × largura × canais. O que nenhum deles faz é **atravessar uma camada**: o que entra e sai de uma `Linear` é sempre vetor ou matriz. A generalidade ainda paga — só que por outro motivo, que aparece no fim desta seção.

Nas bibliotecas de rede neural, um arranjo de *n* dimensões se chama **tensor**, e é assim que vamos chamá-lo também.

> **📌 Nota**
>
> Há razões matemáticas legítimas para não chamar um arranjo de *n* dimensões de "tensor" — em álgebra multilinear, um tensor é um objeto com regras de transformação que um arranjo qualquer não tem. O livro-texto reconhece a objeção e segue em frente, porque o uso da palavra na área já está consolidado. Se você for desses puristas, considere a objeção registrada.

### A trapaça

Um livro dedicado a deep learning implementaria uma classe `Tensor` completa, sobrecarregando os operadores aritméticos do Python e cuidando de uma variedade de operações. Isso ocuparia um capítulo inteiro sozinho. Aqui vamos trapacear:

In [ ]:
Tensor = list

É só isso. Um tensor **é** uma lista.

A trapaça vale numa direção só. Todos os nossos vetores, matrizes e análogos de dimensão mais alta são listas — isso é verdade. Mas a recíproca é falsa: a maioria das listas do Python não é um arranjo de *n* dimensões no sentido que queremos.

O que gostaríamos de escrever é uma definição recursiva:

```python
# Um tensor é ou um float, ou uma lista de tensores
Tensor = Union[float, List[Tensor]]
```

O Python não aceita tipos recursivos definidos assim. E, mesmo que aceitasse, a definição ainda estaria errada: ela permitiria "tensores" como

```python
[[1.0, 2.0],
 [3.0]]
```

cujas linhas têm tamanhos diferentes — o que não é um arranjo de *n* dimensões coisa nenhuma.

> **🔷 Conceito**
>
> Um tensor não é uma estrutura de dados nova. É uma **convenção sobre a forma** de uma lista aninhada: todas as sublistas de um mesmo nível têm o mesmo comprimento, e o aninhamento tem profundidade uniforme.
>
> Nada no Python garante essa convenção, e nós não vamos verificá-la. Tudo o que vem depois — camadas, gradientes, redes inteiras — é recursão sobre listas aninhadas que **presumimos** bem formadas.

### A forma

A primeira função auxiliar descobre a forma de um tensor:

In [ ]:
from typing import List

def shape(tensor: Tensor) -> List[int]:
    sizes: List[int] = []
    while isinstance(tensor, list):
        sizes.append(len(tensor))
        tensor = tensor[0]
    return sizes

assert shape([1, 2, 3]) == [3]
assert shape([[1, 2], [3, 4], [5, 6]]) == [3, 2]

Ela desce pelo primeiro elemento, anotando o comprimento de cada nível, até encontrar algo que não é lista.

> **⚠️ Atenção — `shape` acredita no primeiro elemento**
>
> Repare no `tensor = tensor[0]`: a função olha **só** o primeiro elemento de cada nível. Ela nunca confere se os irmãos dele têm o mesmo tamanho.

In [ ]:
shape([[1, 2], [3]])

> O resultado é `[2, 2]`, e a segunda linha tem um elemento só. Nenhum erro, nenhum aviso — a função devolve uma forma que o objeto não tem.
>
> Isso não é descuido do livro-texto; é a consequência direta de `Tensor = list`. Sem uma classe de verdade, não há lugar onde a invariante possa ser verificada uma vez e valer para sempre. O preço da trapaça é que a corretude do formato vira responsabilidade de quem escreve o código. Guarde isso: é exatamente o tipo de erro que uma biblioteca real pega no construtor, e a nossa não pega em lugar nenhum.

### Recursão, quatro vezes

Como um tensor pode ter qualquer número de dimensões, quase tudo que se faz com ele é recursivo: um caso base para o vetor, uma chamada recursiva para o resto. O teste do caso base é este:

In [ ]:
def is_1d(tensor: Tensor) -> bool:
    """
    Se tensor[0] é uma lista, é um tensor de ordem mais alta.
    Caso contrário, tensor tem uma dimensão só (isto é, é um vetor).
    """
    return not isinstance(tensor[0], list)

assert is_1d([1, 2, 3])
assert not is_1d([[1, 2], [3, 4]])

Com ele, somar todos os valores de um tensor:

In [ ]:
def tensor_sum(tensor: Tensor) -> float:
    """Soma todos os valores do tensor"""
    if is_1d(tensor):
        return sum(tensor)                   # só uma lista de floats: sum do Python
    else:
        return sum(tensor_sum(tensor_i)      # chama tensor_sum em cada linha
                   for tensor_i in tensor)   # e soma os resultados

assert tensor_sum([1, 2, 3]) == 6
assert tensor_sum([[1, 2], [3, 4]]) == 10

Esse é o padrão que vai se repetir o capítulo inteiro. Para não reescrevê-lo toda vez, o livro-texto o encapsula em duas funções de ordem superior. A primeira aplica uma função elemento a elemento:

In [ ]:
from typing import Callable

def tensor_apply(f: Callable[[float], float], tensor: Tensor) -> Tensor:
    """Aplica f elemento a elemento"""
    if is_1d(tensor):
        return [f(x) for x in tensor]
    else:
        return [tensor_apply(f, tensor_i) for tensor_i in tensor]

assert tensor_apply(lambda x: x + 1, [1, 2, 3]) == [2, 3, 4]
assert tensor_apply(lambda x: 2 * x, [[1, 2], [3, 4]]) == [[2, 4], [6, 8]]

E com ela sai de graça uma função que cria um tensor de zeros com a mesma forma de outro:

In [ ]:
def zeros_like(tensor: Tensor) -> Tensor:
    return tensor_apply(lambda _: 0.0, tensor)

assert zeros_like([1, 2, 3]) == [0, 0, 0]
assert zeros_like([[1, 2], [3, 4]]) == [[0, 0], [0, 0]]

A segunda função de ordem superior aplica uma função a elementos **correspondentes** de dois tensores, que deveriam ter exatamente a mesma forma — embora não vamos verificar isso:

In [ ]:
def tensor_combine(f: Callable[[float, float], float],
                   t1: Tensor,
                   t2: Tensor) -> Tensor:
    """Aplica f a elementos correspondentes de t1 e t2"""
    if is_1d(t1):
        return [f(x, y) for x, y in zip(t1, t2)]
    else:
        return [tensor_combine(f, t1_i, t2_i)
                for t1_i, t2_i in zip(t1, t2)]

import operator
assert tensor_combine(operator.add, [1, 2, 3], [4, 5, 6]) == [5, 7, 9]
assert tensor_combine(operator.mul, [1, 2, 3], [4, 5, 6]) == [4, 10, 18]

> **⚠️ Atenção — O `zip` que engole a diferença**
>
> O "deveriam ter exatamente a mesma forma" acima tem consequência prática, e ela vem do `zip`: quando os dois tensores têm tamanhos diferentes, ele simplesmente para no menor.

In [ ]:
tensor_combine(operator.add, [1, 2, 3], [4, 5])

> Somar um tensor de três elementos com um de dois devolve um tensor de dois, sem reclamar. Num laço de treino, isso é uma forma silenciosa de perder um pedaço do gradiente — o modelo continua treinando, um pouco pior, e nada indica o motivo.
>
> Essas duas armadilhas — `shape` que acredita no primeiro elemento e `tensor_combine` que trunca — são o mesmo defeito visto de dois ângulos: **não existe um lugar onde a forma de um tensor seja um fato verificado.** É a primeira coisa que uma biblioteca de verdade acrescenta.

### Seis funções, e o capítulo inteiro

Vale parar um instante e reparar em quão pouco código foi escrito: `shape`, `is_1d`, `tensor_sum`, `tensor_apply`, `zeros_like` e `tensor_combine`. Nada além de recursão sobre listas.

Todas as camadas das próximas seções vão ser escritas com essas peças. A camada sigmoide é um `tensor_apply` na ida e um `tensor_combine` na volta. O gradiente descendente é um `tensor_combine` entre parâmetros e gradientes. O momento é um `zeros_like` mais dois `tensor_combine`. A biblioteca inteira que este capítulo constrói cabe sobre essas seis funções.

E aqui está a dor concreta que a recursão paga, sem precisar de imagem colorida nenhuma. O otimizador da [seção 16.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/05-perda-e-otimizacao.html) percorre `zip(layer.params(), layer.grads())` e aplica o **mesmo** `tensor_combine` a cada par que encontra. Numa camada `Linear(784, 30)`, esses pares são a matriz de pesos, de forma `[30, 784]`, **e** o vetor de vieses, de forma `[30]`: duas dimensões diferentes, no mesmo laço, atendidas pela mesma chamada. Não há um `if` distinguindo os dois casos em lugar nenhum do otimizador — e escrever a atualização de pesos sem recursão significaria ou dois laços, ou esse `if`, ou obrigar toda camada a expor os parâmetros na mesma forma. É esse o payoff da generalidade neste livro, e ele chega três seções antes de qualquer lote de imagens.

In [ ]:
# a rede XOR do capítulo 15, na forma em que ela existia lá
xor_network = [[[20., 20, -30],
                [20., 20, -10]],
               [[-60., 60, -30]]]

print("forma da rede do capítulo 15:", shape(xor_network))
print("total de pesos:", tensor_sum(tensor_apply(lambda _: 1.0, xor_network)))

Aquela "lista de camadas de neurônios de pesos" que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) descreveu em português tem, agora, um nome e uma forma medida: é um tensor irregular de forma `[2, 2, 3]` — irregular porque a segunda camada tem um neurônio, não dois, e `shape` acreditou no primeiro elemento, como avisado acima. Isso é um bom retrato do que muda neste capítulo: a partir da próxima seção, os pesos deixam de morar numa lista aninhada solta e passam a morar **dentro de uma camada**, que sabe o que fazer com eles.

> **💡 Dica — Na prática: `numpy.ndarray` e `torch.Tensor`**
>
> O `Tensor = list` deste livro é o único lugar da área onde um tensor é uma lista aninhada. Em qualquer biblioteca real, um tensor é um objeto com três coisas que a nossa lista não tem:
>
> **Um bloco contíguo de memória.** Um `numpy.ndarray` de 60.000 × 784 floats é um único bloco de bytes, com um tipo (`dtype`) e um passo (`strides`) por dimensão. A nossa lista de listas é um vetor de ponteiros para vetores de ponteiros para objetos `float` do Python, cada um com cabeçalho, contador de referências e alocação própria. Percorrer o primeiro é sequencial; percorrer o segundo é perseguir ponteiros pela memória inteira.
>
> **Uma forma que é um fato.** `array.shape` não desce pelo primeiro elemento adivinhando: é um atributo, fixado na construção. Arranjos irregulares simplesmente não existem — a construção falha. As duas caixas de aviso desta seção não têm equivalente lá.
>
> **Operações no laço interno em código compilado.** `a + b` sobre dois arranjos de um milhão de elementos é um laço em C, possivelmente vetorizado pelo processador; num `torch.Tensor` em GPU, são milhares de somas simultâneas. O nosso `tensor_combine` é um `zip` interpretado, um elemento por vez.
>
> A diferença de velocidade entre as duas coisas é de duas a quatro ordens de grandeza, e a [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) mede exatamente o que isso significa em minutos de espera.
>
> **E ainda assim não vamos usar nenhuma delas.** Reescrever este capítulo com `numpy` deixaria o código mais curto e centenas de vezes mais rápido, e apagaria o assunto: a recursão explícita acima **é** o que uma biblioteca de tensores faz, e vê-la escrita em seis funções de três linhas é o que permite reconhecê-la depois, escondida atrás de um operador sobrecarregado.

## A Abstração de Camada

> **📌 Nota**
>
> Esta seção corresponde a *The Layer Abstraction*, do capítulo 19 de Grus (2019).

Esta seção só faz sentido depois de doer, então vale recuperar a dor.

O [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) construiu uma rede que empilhava duas camadas de neurônios, cada um calculando `sigmoid(dot(weights, inputs))`. O `feed_forward` de lá, aliás, aceitava qualquer número de camadas. O problema não estava na ida — estava na volta. A função `sqerror_gradients`, da [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html), começava assim:

```python
hidden_outputs, outputs = feed_forward(network, input_vector)
```

Essa linha desempacota a lista de saídas em exatamente **dois** nomes. Uma rede de três camadas estoura ali com `ValueError`. E o resto da função — `network[0]`, `network[-1]`, `output_deltas`, `hidden_deltas` — supõe o tempo todo que existe uma camada escondida e uma de saída, e nada entre elas.

Havia uma segunda rigidez, mais escondida. A sigmoide não estava em lugar nenhum como uma escolha: ela estava soldada dentro de `neuron_output` na ida, e a derivada dela aparecia escrita à mão, como `output * (1 - output)`, em duas linhas diferentes da passada para trás. **Trocar a função de ativação significava reescrever o gradiente à mão**, em pontos do código que nem se pareciam entre si.

> **🔷 Conceito**
>
> Os dois problemas têm a mesma raiz: o `forward` e o `backward` estavam **entrelaçados num laço só**, escrito para uma topologia específica.
>
> A abstração desta seção separa as duas coisas na única fronteira em que elas se separam bem: a camada. Cada camada passa a saber fazer a sua própria conta na ida **e** propagar o gradiente através de si mesma na volta. Ninguém mais precisa saber quantas camadas existem, nem qual função cada uma aplica.

### O contrato

Uma camada é qualquer coisa que saiba fazer quatro coisas:

In [ ]:
from typing import Iterable, Tuple, List

Tensor = list

class Layer:
    """
    Nossas redes neurais serão compostas de Layers, cada uma sabendo
    fazer alguma conta sobre suas entradas na direção "forward" e
    propagar gradientes na direção "backward".
    """
    def forward(self, input):
        """
        Repare na falta de tipos. Não vamos ser prescritivos sobre
        que tipos de entrada as camadas podem receber nem que tipos
        de saída podem devolver.
        """
        raise NotImplementedError

    def backward(self, gradient):
        """
        Do mesmo modo, não vamos ser prescritivos sobre a forma do
        gradiente. Cabe a você, usuário, garantir que está fazendo
        coisas sensatas.
        """
        raise NotImplementedError

    def params(self) -> Iterable[Tensor]:
        """
        Devolve os parâmetros desta camada. A implementação padrão
        não devolve nada, para que uma camada sem parâmetros não
        precise implementar isto.
        """
        return ()

    def grads(self) -> Iterable[Tensor]:
        """
        Devolve os gradientes, na mesma ordem de params().
        """
        return ()

Quatro métodos, e dois deles com implementação padrão vazia. Vale ler o que cada um significa:

- **`forward(input)`** calcula a saída da camada a partir da entrada. É a metade fácil.
- **`backward(gradient)`** recebe $\partial L / \partial \text{saída}$ e devolve $\partial L / \partial \text{entrada}$. Isto é, recebe a culpa que chegou da camada seguinte e devolve a culpa que cabe à camada anterior. É a regra da cadeia da [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html), agora encapsulada num método.
- **`params()`** devolve os tensores de parâmetros da camada, se houver. Uma camada de ativação não tem nenhum, e por isso o padrão devolve a tupla vazia.
- **`grads()`** devolve os gradientes correspondentes, **na mesma ordem**. Essa correspondência posicional é o que permite ao otimizador percorrer os dois em paralelo sem saber nada sobre a camada.

> **📌 Nota — A falta de anotações de tipo é deliberada**
>
> O livro-texto anota tipos com rigor em quase tudo, e aqui abre exceção: `forward` e `backward` não dizem o que recebem nem o que devolvem.
>
> O motivo é que camadas diferentes trabalham com formas diferentes. Uma camada linear recebe um vetor; uma camada convolucional receberia um tensor de três dimensões; uma camada recorrente carregaria estado entre chamadas. Prescrever um tipo aqui limitaria o que se pode encaixar depois.
>
> O preço é que nada verifica se a saída de uma camada faz sentido como entrada da próxima. O livro-texto diz isso na cara: *cabe a você garantir que está fazendo coisas sensatas*. É a mesma ausência de verificação da seção anterior, agora na fronteira entre camadas em vez de dentro do tensor.

### A camada mais simples que existe

Uma forma de olhar as redes do capítulo anterior é como uma camada "linear", seguida de uma camada "sigmoide", seguida de outra linear e outra sigmoide. Não as distinguíamos nesses termos, e é justamente essa distinção que permite experimentar estruturas mais gerais.

A camada sigmoide não tem parâmetro nenhum: ela só aplica uma função a cada elemento.

In [ ]:
from scratch.deep_learning import tensor_apply, tensor_combine
from scratch.neural_networks import sigmoid

In [ ]:
class Sigmoid(Layer):
    def forward(self, input: Tensor) -> Tensor:
        """
        Aplica a sigmoide a cada elemento do tensor de entrada,
        e guarda os resultados para usar na retropropagação.
        """
        self.sigmoids = tensor_apply(sigmoid, input)
        return self.sigmoids

    def backward(self, gradient: Tensor) -> Tensor:
        return tensor_combine(lambda sig, grad: sig * (1 - sig) * grad,
                              self.sigmoids,
                              gradient)

> **📌 Nota**
>
> As funções `tensor_apply` e `tensor_combine` são as da [seção 16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-o-tensor.html) — importadas de `scratch.deep_learning`, o pacote copiado literalmente do repositório do livro-texto, e não uma segunda versão delas. Cada página deste livro roda num kernel próprio, então nomes definidos numa página não existem em outra; o pacote existe em todas. A `sigmoid` vem de `scratch.neural_networks`, escrita no [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html).
>
> O chunk do `import` leva `#| output: false` por um motivo específico deste projeto: `scratch/deep_learning.py` importa `scratch/probability.py`, que tem dezoito chamadas `plt.*` soltas no nível do módulo. O import deixa uma figura pendurada, e sem o `output: false` ela apareceria como saída da célula. Nada disso se corrige editando `scratch/` — o pacote é vendorizado e não se edita.

Três coisas para reparar nessa classe.

**A primeira é `self.sigmoids`.** Na ida, a camada guarda o que calculou. Na volta, ela precisa desses valores: a derivada da sigmoide em um ponto se escreve em função da própria saída, $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$. Praticamente toda camada vai fazer esse tipo de coisa — guardar, na passada para a frente, o que a passada para trás precisar.

**A segunda é de onde vem o `sig * (1 - sig) * grad`.** É a regra da cadeia, e ela corresponde exatamente ao termo `output * (1 - output) * (output - target)` da rede do capítulo anterior. O que mudou é o lugar: lá o `(output - target)` vinha da perda e estava misturado com a derivada da ativação na mesma expressão. Aqui a camada recebe o `grad` de fora — ela não sabe nem se importa de onde ele veio — e só multiplica pela sua própria derivada.

**A terceira é que `params` e `grads` não foram implementados.** A camada não tem o que atualizar, então a implementação padrão da classe-base serve, e o otimizador simplesmente não terá nada a fazer com ela.

### Conferindo

A camada funciona? A ida é fácil de conferir de cabeça; a volta merece uma comparação com a derivada calculada à mão. Passando um gradiente de 1 em toda parte, `backward` deveria devolver exatamente $\sigma(x)(1-\sigma(x))$ para cada entrada $x$:

In [ ]:
camada = Sigmoid()
entrada = [[-2.0, 0.0], [1.0, 3.0]]

saida = camada.forward(entrada)
print("forward       :", [[round(v, 4) for v in linha] for linha in saida])

grad = camada.backward([[1.0, 1.0], [1.0, 1.0]])
print("backward(1s)  :", [[round(v, 4) for v in linha] for linha in grad])
print("σ'(x) à mão   :", [[round(sigmoid(x) * (1 - sigmoid(x)), 4)
                           for x in linha] for linha in entrada])

As duas últimas linhas coincidem, casa por casa. Repare também no maior valor da derivada, 0,25, que ocorre na entrada 0 — é o pico do sino que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) desenhou. E repare no menor, 0,0452 para a entrada 3: um neurônio com pré-ativação 3 já está quase saturado, e o gradiente que passa por ele é cinco vezes menor. Esse é o problema que a [seção 16.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/06-outras-funcoes-de-ativacao.html) vai atacar trocando a ativação — o que agora custa trocar um objeto de lugar, e não reescrever gradiente nenhum.

> **⚠️ Atenção — A camada guarda estado, e ninguém confere a ordem**
>
> `backward` só funciona se `forward` tiver sido chamado antes, com a entrada certa. A camada não verifica isso.

In [ ]:
outra = Sigmoid()
outra.forward([0.0, 0.0])          # a camada guarda [0.5, 0.5]
outra.forward([10.0, 10.0])        # agora guarda [0.99995, 0.99995]
outra.backward([1.0, 1.0])         # gradiente de QUAL das duas passadas?

> O resultado é minúsculo, e não 0,25: é a derivada da sigmoide em 10, não em 0. O `self.sigmoids` da primeira passada foi sobrescrito pela segunda, e a camada retropropagou a que sobrou. Se o seu laço de treino calcular duas previsões antes de retropropagar a primeira, os gradientes da primeira estão perdidos, e nada avisa: o treino continua, converge para algum lugar, e o modelo simplesmente fica pior do que deveria.
>
> Isso vale para **toda** camada deste capítulo, e é a razão pela qual o laço de treino que vamos escrever sempre faz `forward`, depois `backward`, depois `step`, um exemplo por vez. A ordem não é estilo; é a única ordem correta.

> **💡 Dica — Na prática: `torch.nn.Module`**
>
> A classe que você acabou de escrever tem um equivalente exato, e o nome dele é `torch.nn.Module`. Uma camada em PyTorch é assim:
>
> ```python
> import torch
> import torch.nn as nn
>
> class Sigmoide(nn.Module):
>     def forward(self, x):
>         return 1 / (1 + torch.exp(-x))
> ```
>
> Repare no que **não** está lá: o `backward`. É a diferença mais importante entre a nossa biblioteca e uma de verdade.
>
> O PyTorch não pede que você escreva a passada para trás porque ele a deriva sozinho. Cada operação sobre um tensor registra, num grafo, qual foi a operação e quais foram os operandos; quando você chama `.backward()` no resultado final, o framework percorre esse grafo na ordem inversa aplicando a regra da cadeia. É a **diferenciação automática em modo reverso** que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) mencionou, e ela torna o nosso `backward` desnecessário — o preço é que o grafo precisa existir, o que exige a classe `Tensor` completa que decidimos não escrever.
>
> O resto do contrato está lá, com outros nomes: `params()` virou `.parameters()`, `grads()` virou o atributo `.grad` de cada parâmetro, e a distinção entre camadas com e sem parâmetro é feita por registro automático em vez de por implementação padrão.
>
> Vale notar o contraste com o `scikit-learn`, que apareceu nos callouts do capítulo anterior. O `MLPClassifier` **não** tem essa abstração: ele é um objeto monolítico que recebe `hidden_layer_sizes=(25,)` e `activation='relu'` como argumentos, e não há camada nenhuma para instanciar, compor ou substituir. As duas bibliotecas resolvem problemas diferentes — e a fronteira entre elas é exatamente a abstração desta seção.

## A Camada Linear

> **📌 Nota**
>
> Esta seção corresponde a *The Linear Layer*, do capítulo 19 de Grus (2019).

Falta a outra metade da rede do capítulo anterior: a camada que representa o `dot(weights, inputs)` dos neurônios. Ao contrário da sigmoide, essa camada **tem** parâmetros — e parâmetros precisam ser inicializados.

### A inicialização não é detalhe

Os valores iniciais dos pesos fazem uma diferença enorme na rapidez com que a rede treina, e às vezes em **se** ela treina. O motivo é a saturação que a seção anterior mediu: se os pesos forem grandes demais, as pré-ativações caem numa faixa onde a derivada da ativação é quase zero, e uma parte da rede com gradiente zero não aprende nada por gradiente descendente — nunca.

O livro-texto registra isso com uma franqueza que vale citar: *algumas das redes deste capítulo eu não consegui treinar de jeito nenhum com inicializações diferentes das que usei.*

São três esquemas. O primeiro sorteia cada valor uniformemente em $[0, 1]$, com `random.random()`. O segundo, que é o padrão, sorteia de uma normal padrão. O terceiro é a **inicialização de Xavier**, em que cada peso vem de uma normal de média 0 e variância $2 / (\text{entradas} + \text{saídas})$ — uma escolha que, segundo o livro-texto, costuma funcionar bem para pesos de rede neural.

In [ ]:
import random
from typing import List, Iterable
from scratch.deep_learning import Layer, Tensor
from scratch.probability import inverse_normal_cdf
from scratch.linear_algebra import dot

> **📌 Nota**
>
> O `Layer` e o `Tensor` vêm de `scratch.deep_learning` — são o mesmo código da [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-a-abstracao-de-camada.html), não uma segunda versão dele, já que cada página deste livro roda num kernel próprio. O `inverse_normal_cdf` vem de `scratch/probability.py`, módulo do capítulo de probabilidade do livro-texto, que ficou fora da ementa desta disciplina mas do qual o pacote depende — e cujo import desenha uma figura, pelo motivo que a [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-a-abstracao-de-camada.html) explica. Daí o `#| output: false`.

In [ ]:
def random_uniform(*dims: int) -> Tensor:
    if len(dims) == 1:
        return [random.random() for _ in range(dims[0])]
    else:
        return [random_uniform(*dims[1:]) for _ in range(dims[0])]

def random_normal(*dims: int,
                  mean: float = 0.0,
                  variance: float = 1.0) -> Tensor:
    if len(dims) == 1:
        return [mean + variance * inverse_normal_cdf(random.random())
                for _ in range(dims[0])]
    else:
        return [random_normal(*dims[1:], mean=mean, variance=variance)
                for _ in range(dims[0])]

from scratch.deep_learning import shape

assert shape(random_uniform(2, 3, 4)) == [2, 3, 4]
assert shape(random_normal(5, 6, mean=10)) == [5, 6]

As duas funções seguem o mesmo padrão recursivo da [seção 16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-o-tensor.html): o caso base produz uma lista de números, e o caso geral produz uma lista de tensores de uma dimensão a menos. Os três esquemas se embrulham numa função só:

In [ ]:
def random_tensor(*dims: int, init: str = 'normal') -> Tensor:
    if init == 'normal':
        return random_normal(*dims)
    elif init == 'uniform':
        return random_uniform(*dims)
    elif init == 'xavier':
        variance = len(dims) / sum(dims)
        return random_normal(*dims, variance=variance)
    else:
        raise ValueError(f"unknown init: {init}")

O `len(dims) / sum(dims)` é a fórmula de Xavier escrita de um jeito compacto: para um tensor de duas dimensões, `len(dims)` vale 2 e `sum(dims)` vale entradas mais saídas.

> **⚠️ Atenção — O parâmetro chamado `variance` é o desvio padrão**
>
> Olhe a linha que faz o sorteio:
>
> ```python
> mean + variance * inverse_normal_cdf(random.random())
> ```
>
> O `inverse_normal_cdf(u)` devolve um valor sorteado de uma **normal padrão**, de variância 1. Multiplicar uma normal padrão por uma constante $c$ produz uma normal de desvio padrão $c$ — e portanto de variância $c^2$, não $c$. O parâmetro se chama `variance`, mas o papel que ele desempenha na conta é o de desvio padrão.
>
> Dá para medir:

In [ ]:
import statistics

random.seed(0)
amostra = random_normal(100000, variance=4.0)

print(f"desvio padrão da amostra: {statistics.pstdev(amostra):.4f}")
print(f"variância da amostra:     {statistics.pvariance(amostra):.4f}")

> Pedimos variância 4 e recebemos variância 16, que é $4^2$.
>
> A consequência para a inicialização de Xavier é maior do que parece, porque a fórmula real de Glorot e Bengio pede **desvio padrão** $\sqrt{2/(n_{ent} + n_{sai})}$, e o código passa $2/(n_{ent} + n_{sai})$ como se fosse o desvio:

In [ ]:
import math

random.seed(0)
pesos = random_tensor(30, 784, init='xavier')
achatado = [w for linha in pesos for w in linha]

print(f"desvio padrão obtido: {statistics.pstdev(achatado):.5f}")
print(f"2 / (784 + 30)      = {2 / 814:.5f}")
print(f"sqrt(2 / (784 + 30)) = {math.sqrt(2 / 814):.5f}")

> Os pesos saem cerca de **vinte vezes mais estreitos** do que a fórmula de Xavier pediria. Não vamos consertar isso — `scratch/` é uma cópia literal do repositório do livro-texto e não se edita, e além disso as redes deste capítulo treinam assim, com estes valores. Registre como aviso de leitura: quando o livro-texto diz "inicialização de Xavier", o que o código faz não é bem isso, e a observação dele sobre não conseguir treinar com outras inicializações provavelmente tem a ver com esta discrepância.

### A camada

Com o sorteio resolvido, a camada linear precisa saber a dimensão da entrada (quantos pesos cada neurônio tem), a dimensão da saída (quantos neurônios são) e o esquema de inicialização:

In [ ]:
class Linear(Layer):
    def __init__(self, input_dim: int, output_dim: int,
                 init: str = 'xavier') -> None:
        """
        Uma camada de output_dim neurônios, cada um com input_dim pesos
        (e um viés).
        """
        self.input_dim = input_dim
        self.output_dim = output_dim

        # self.w[o] são os pesos do o-ésimo neurônio
        self.w = random_tensor(output_dim, input_dim, init=init)

        # self.b[o] é o termo de viés do o-ésimo neurônio
        self.b = random_tensor(output_dim, init=init)

    def forward(self, input: Tensor) -> Tensor:
        # Guarda a entrada para usar na passada para trás.
        self.input = input

        # Devolve o vetor de saídas dos neurônios.
        return [dot(input, self.w[o]) + self.b[o]
                for o in range(self.output_dim)]

    def backward(self, gradient: Tensor) -> Tensor:
        # Cada b[o] é somado a output[o], o que significa que
        # o gradiente de b é igual ao gradiente da saída.
        self.b_grad = gradient

        # Cada w[o][i] multiplica input[i] e é somado a output[o].
        # Então o gradiente dele é input[i] * gradient[o].
        self.w_grad = [[self.input[i] * gradient[o]
                        for i in range(self.input_dim)]
                       for o in range(self.output_dim)]

        # Cada input[i] multiplica todo w[o][i] e é somado a todo
        # output[o]. Então o gradiente dele é a soma de w[o][i] * gradient[o]
        # sobre todas as saídas.
        return [sum(self.w[o][i] * gradient[o] for o in range(self.output_dim))
                for i in range(self.input_dim)]

    def params(self) -> Iterable[Tensor]:
        return [self.w, self.b]

    def grads(self) -> Iterable[Tensor]:
        return [self.w_grad, self.b_grad]

> **🔷 Conceito — A dívida do Capítulo 4, cobrada**
>
> A [seção 4.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/02-matrizes.html) prometeu que uma matriz $n \times k$ representa uma função linear que leva vetores de $k$ dimensões em vetores de $n$ dimensões, e disse que as camadas de uma rede neural são exatamente isso. Aqui está a promessa, em código: `self.w` é uma matriz de forma `[output_dim, input_dim]`, e `forward` é a multiplicação dessa matriz pelo vetor de entrada, mais um deslocamento.
>
> Repare numa mudança em relação ao [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html): lá, o viés era anexado ao fim do vetor de pesos e a entrada ganhava um 1 constante, para que a conta do neurônio virasse um produto escalar puro. **Aqui o viés é um tensor separado, `self.b`.** Aquilo era contabilidade nossa, não uma verdade sobre redes neurais — e o callout do capítulo anterior já tinha observado que o `scikit-learn` também guarda o viés à parte, em `intercepts_`. As duas representações descrevem a mesma função.

O `forward` é a metade fácil: uma saída por neurônio, cada uma um produto escalar mais o viés, e o guardar da entrada para depois. O `backward` é a parte que dá trabalho, e ele tem **três parcelas** — as mesmas três que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) descreveu em português, agora escritas uma vez e para sempre. Elas se leem assim:

- **O viés recebe o gradiente inteiro.** `b[o]` entra somado em `output[o]` com coeficiente 1, então $\partial \text{saída}[o] / \partial b[o] = 1$ e a culpa passa direto.
- **A culpa de um peso é a culpa do neurônio vezes a entrada que aquele peso multiplica.** É a mesma frase da [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html): um peso cuja entrada valia 0 naquele exemplo não teve influência nenhuma, e o gradiente dele é zero.
- **A culpa de uma entrada é emprestada de todos os neurônios que a usaram.** Cada `input[i]` foi enviada a todos os `output_dim` neurônios, cada um por um peso conhecido; a culpa volta por esses mesmos pesos e se soma. É o "o mesmo peso que levou o sinal para a frente traz a culpa de volta", agora sem a derivada da sigmoide junto — porque a ativação virou outra camada.

E `params` e `grads` são a razão de existirem: a camada expõe dois tensores de parâmetros e dois de gradiente, **na mesma ordem**, e o otimizador da [seção 16.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/05-perda-e-otimizacao.html) vai percorrer os dois em paralelo sem saber o que eles são.

### Conferindo o gradiente por diferenças finitas

Um `backward` errado não estoura: ele treina mal. Vale conferir, e a forma padrão de conferir é comparar o gradiente analítico com a derivada numérica.

A ideia é direta. Se $g$ é o gradiente que chega de cima, a perda que estamos derivando é $L = \sum_o g[o] \cdot \text{saída}[o]$ — porque $\partial L/\partial \text{saída} = g$, por construção. Então basta empurrar uma entrada um tiquinho para cada lado e medir quanto $L$ muda:

In [ ]:
random.seed(0)

camada = Linear(3, 2)
x = [0.5, -1.0, 2.0]
g = [1.0, -2.0]                      # gradiente vindo de cima, escolhido a dedo

camada.forward(x)
grad_entrada = camada.backward(g)

def perda(entrada):
    saida = camada.forward(entrada)
    return sum(a * b for a, b in zip(saida, g))

h = 1e-6
numerico = []
for i in range(3):
    mais, menos = list(x), list(x)
    mais[i] += h
    menos[i] -= h
    numerico.append((perda(mais) - perda(menos)) / (2 * h))

print("analítico:", [round(v, 6) for v in grad_entrada])
print("numérico :", [round(v, 6) for v in numerico])

Os dois batem em todas as casas impressas. Vale fazer o mesmo com um peso, para conferir a segunda parcela:

In [ ]:
camada.forward(x)
camada.backward(g)

o, i = 1, 2                          # o peso do 2º neurônio para a 3ª entrada
original = camada.w[o][i]

camada.w[o][i] = original + h; mais = perda(x)
camada.w[o][i] = original - h; menos = perda(x)
camada.w[o][i] = original

print(f"analítico: {camada.w_grad[o][i]:.6f}")
print(f"numérico : {(mais - menos) / (2 * h):.6f}")

O valor é exatamente `input[2] * g[1]`, isto é, $2{,}0 \times (-2{,}0) = -4$, como a segunda parcela prometia.

> **🟩 Exemplo — Por que essa conferência importa mais aqui do que em outros capítulos**
>
> Nos capítulos anteriores, um gradiente errado aparecia: a perda não caía, o `assert` do fim falhava, o modelo previa besteira.
>
> Numa rede de várias camadas, um `backward` sutilmente errado numa camada **do meio** costuma não fazer nada disso. A rede ainda treina, porque as outras camadas compensam; a perda ainda cai, mais devagar; e o resultado final é um pouco pior, dentro da faixa que se atribuiria a "escolhi mal a taxa de aprendizado". A conferência por diferenças finitas é o teste que separa as duas hipóteses, e é padrão em qualquer implementação de camada nova — inclusive nas bibliotecas de verdade, que trazem utilitários prontos para isso.

### Quantos parâmetros

Uma última observação de escala, que vai importar na [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html):

In [ ]:
for entradas, saidas in [(2, 2), (10, 25), (784, 30), (784, 10)]:
    print(f"Linear({entradas}, {saidas}):"
          f" {entradas * saidas} pesos + {saidas} vieses"
          f" = {entradas * saidas + saidas} parâmetros")

Uma camada que recebe uma imagem de 28 × 28 pixels e produz 30 números tem 23.550 parâmetros. Compare com os 4 coeficientes da regressão múltipla do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), que se liam um a um em português, e com os 379 pesos da rede do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html).

> **💡 Dica — Na prática: `torch.nn.Linear`**
>
> A classe que você acabou de escrever existe no PyTorch com o mesmo nome e quase a mesma assinatura:
>
> ```python
> import torch.nn as nn
>
> camada = nn.Linear(in_features=784, out_features=30)
> camada.weight     # tensor de forma (30, 784)
> camada.bias       # tensor de forma (30,)
> ```
>
> `weight` tem exatamente a forma do nosso `self.w`, e `bias` a do nosso `self.b`. A conta é a mesma: $Wx + b$.
>
> Três diferenças, todas informativas.
>
> **A multiplicação é uma operação só.** O nosso `forward` é uma compreensão de lista que chama `dot` uma vez por neurônio, e cada `dot` é um `sum` sobre um `zip`. O `nn.Linear` faz uma multiplicação de matriz, despachada para uma rotina de álgebra linear compilada — e, se houver GPU, para milhares de núcleos ao mesmo tempo. Além disso, ele processa um **lote** de exemplos de uma vez: a entrada é uma matriz de forma (exemplos × entradas), e a saída sai numa multiplicação só. O nosso laço de treino processa um exemplo por vez, porque a nossa camada não sabe fazer outra coisa.
>
> **A inicialização padrão é outra, e o nome dela engana.** O nosso padrão é `init='xavier'`, com a ressalva medida acima. No código-fonte do PyTorch, a `nn.Linear` inicializa com `kaiming_uniform_(weight, a=sqrt(5))` — o que se lê, à primeira vista, como "a inicialização de Kaiming, aquela pensada para a ReLU". **Não é.** O parâmetro $a = \sqrt5$ é o ganho, e ele cancela exatamente o fator que a versão para ReLU acrescenta: o limite do sorteio uniforme cai de $\sqrt{6/\text{entradas}}$ para $1/\sqrt{\text{entradas}}$. O próprio comentário no fonte diz isso com todas as letras, e a documentação da classe descreve apenas o resultado — pesos sorteados em $\mathcal{U}(-\sqrt{k}, \sqrt{k})$, com $k = 1/\texttt{in\_features}$ —, sem mencionar ReLU em lugar nenhum.
>
> É um bom exemplo de uma coisa que este livro repete: ler o nome de uma função não substitui ler a fórmula que ela executa.
>
> **O gradiente não é seu problema.** Não existe um `backward` a implementar: as três parcelas que escrevemos à mão saem do grafo de autodiferenciação. Elas continuam sendo calculadas — só que por código gerado, e não por código escrito.
>
> O que a biblioteca **não** faz diferente é a matemática. `w_grad[o][i] = input[i] * gradient[o]` é literalmente o que acontece lá dentro, para cada um dos 23.550 parâmetros.

## Redes como Sequência de Camadas

> **📌 Nota**
>
> Esta seção corresponde a *Neural Networks as a Sequence of Layers*, do capítulo 19 de Grus (2019).

Temos duas camadas: a `Linear`, com parâmetros, e a `Sigmoid`, sem. Falta o que junta várias delas numa rede.

A resposta é curta, e o truque está em enxergá-la: **a rede resultante é ela mesma uma camada**. Ela recebe uma entrada e devolve uma saída; ela recebe um gradiente e devolve um gradiente; ela tem parâmetros, que são os das camadas de dentro. Então ela implementa a interface `Layer`, e implementa cada método da forma óbvia:

In [ ]:
import random
from typing import List, Iterable
from scratch.deep_learning import Layer, Linear, Sigmoid, Tensor, shape

In [ ]:
class Sequential(Layer):
    """
    Uma camada que consiste numa sequência de outras camadas.
    Cabe a você garantir que a saída de cada camada faz sentido
    como entrada da camada seguinte.
    """
    def __init__(self, layers: List[Layer]) -> None:
        self.layers = layers

    def forward(self, input):
        """Só passa a entrada pelas camadas, em ordem."""
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def backward(self, gradient):
        """Só retropropaga o gradiente pelas camadas, na ordem inversa."""
        for layer in reversed(self.layers):
            gradient = layer.backward(gradient)
        return gradient

    def params(self) -> Iterable[Tensor]:
        """Só devolve os params de cada camada."""
        return (param for layer in self.layers for param in layer.params())

    def grads(self) -> Iterable[Tensor]:
        """Só devolve os grads de cada camada."""
        return (grad for layer in self.layers for grad in layer.grads())

Quatro métodos, quatro linhas de corpo cada. E a palavra que aparece em todos os quatro comentários — *só* — é o ponto da seção.

> **🔷 Conceito**
>
> `forward` é um laço para a frente. `backward` é o **mesmo** laço com `reversed`. É a assimetria inteira entre a ida e a volta de uma rede neural, escrita como uma chamada de função embutida.
>
> Isso não é uma coincidência de implementação. A retropropagação é a regra da cadeia aplicada na ordem inversa da composição, e composição de funções é exatamente o que `Sequential` representa. O `reversed` é a regra da cadeia.

Repare também no que a classe **não** faz: ela não sabe quantas camadas tem, nem que tipo elas são, nem se elas têm parâmetros. O `params()` de uma camada de ativação devolve a tupla vazia, e o gerador simplesmente não produz nada para ela. Nenhum `if`, em lugar nenhum.

### A rede XOR, nesta representação

A rede do XOR, que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-redes-feed-forward.html) escreveu como uma lista de listas de listas, agora se monta assim:

In [ ]:
random.seed(0)

xor_net = Sequential([
    Linear(input_dim=2, output_dim=2),
    Sigmoid(),
    Linear(input_dim=2, output_dim=1),
    Sigmoid()
])

[shape(param) for param in xor_net.params()]

Quatro tensores de parâmetros: a matriz $2 \times 2$ e o viés da primeira camada linear, a matriz $1 \times 2$ e o viés da segunda. As camadas de sigmoide não contribuem com nada, como esperado.

### É a mesma rede?

Uma forma convincente de mostrar que a nova representação não mudou o modelo é **transplantar** os pesos que o capítulo anterior escolheu à mão para dentro desta estrutura, e comparar as saídas.

Lá, cada neurônio era um vetor com o viés grudado no fim: `[20., 20, -30]` significava pesos 20 e 20 e viés −30. Aqui os dois moram separados:

In [ ]:
from scratch.neural_networks import feed_forward

# a rede escolhida à mão na seção 15.2
xor_network = [[[20., 20, -30],      # neurônio 'e'
                [20., 20, -10]],     # neurônio 'ou'
               [[-60., 60, -30]]]    # neurônio '2ª entrada mas não a 1ª'

# os mesmos números, na representação desta seção
xor_net.layers[0].w = [[20., 20.], [20., 20.]]
xor_net.layers[0].b = [-30., -10.]
xor_net.layers[2].w = [[-60., 60.]]
xor_net.layers[2].b = [-30.]

In [ ]:
for entrada in [[0., 0], [0., 1], [1., 0], [1., 1]]:
    nova = xor_net.forward(entrada)[0]
    velha = feed_forward(xor_network, entrada)[-1][0]
    print(f"{entrada} -> Sequential {nova:.6f}   "
          f"feed_forward do cap. 15 {velha:.6f}   iguais: {nova == velha}")

Bit por bit iguais — não "próximos", **iguais**, porque as duas implementações fazem exatamente as mesmas multiplicações e somas, na mesma ordem. O que mudou foi só onde os números moram e quem sabe derivá-los.

### A profundidade ficou de graça

A [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) terminou com um aviso: aquela implementação da retropropagação só funcionava para duas camadas, e cada topologia nova exigiria uma função nova. Esse aviso acabou de expirar.

In [ ]:
random.seed(0)

funda = Sequential([
    Linear(10, 8), Sigmoid(),
    Linear(8, 6),  Sigmoid(),
    Linear(6, 4),  Sigmoid(),
    Linear(4, 2)
])

print("formas dos parâmetros:", [shape(p) for p in funda.params()])
print("saída para um vetor de 10 zeros:",
      [round(v, 4) for v in funda.forward([0.0] * 10)])

Sete camadas, quatro delas com parâmetros. Nenhuma linha de código de gradiente foi escrita para isso funcionar: o `backward` de `Sequential` percorre a lista ao contrário, e cada camada sabe se virar. A rede acima é "profunda" no sentido original da expressão *deep learning* — mais de uma camada escondida —, e escrevê-la custou uma lista.

> **🟩 Exemplo — Uma rede dentro de uma rede**
>
> Como `Sequential` é uma `Layer`, nada impede pôr uma dentro da outra:

In [ ]:
random.seed(0)

bloco = Sequential([Linear(4, 4), Sigmoid()])

aninhada = Sequential([
    Linear(10, 4),
    bloco,
    bloco,          # o MESMO bloco, duas vezes
    Linear(4, 2)
])

print("parâmetros vistos por params():", [shape(p) for p in aninhada.params()])

> Repare no detalhe: o mesmo objeto `bloco` aparece duas vezes na lista, e por isso os pesos dele aparecem **duplicados** em `params()` — o gerador percorre a lista de camadas, não um conjunto de objetos distintos. E o estrago acontece antes do otimizador entrar em cena: no `backward`, o bloco é visitado duas vezes e escreve `self.w_grad` nas duas, então a segunda visita **sobrescreve** o gradiente da primeira. Quando o `step` chega, as duas entradas de `grads()` devolvem o mesmo tensor — o que sobrou —, e o otimizador aplica duas vezes o mesmo passo ao mesmo parâmetro. Um dos dois gradientes sumiu; o outro conta em dobro.
>
> Compartilhar pesos entre camadas é uma técnica real e útil — é o que faz uma rede recorrente ser recorrente. Mas fazê-la funcionar exige que os gradientes se **acumulem** em vez de se sobrescreverem, e a nossa `Linear` faz `self.w_grad = ...`, não `+=`. Ou seja: a composição permite escrever a rede, e o resto da biblioteca não a treina corretamente. É o custo de uma abstração que não verifica nada — o mesmo custo das duas caixas de aviso da [seção 16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-o-tensor.html).

> **💡 Dica — Na prática: `torch.nn.Sequential` e `keras.Sequential`**
>
> Desta vez não há nem tradução a fazer. O PyTorch expõe:
>
> ```python
> import torch.nn as nn
>
> modelo = nn.Sequential(
>     nn.Linear(784, 30),
>     nn.Tanh(),
>     nn.Linear(30, 10),
> )
> ```
>
> Mesmo nome de classe, mesma ideia, mesma lista de camadas. E no Keras — que na versão 3 roda indiferentemente sobre TensorFlow, JAX ou o próprio PyTorch — é a mesma frase de novo:
>
> ```python
> import keras
>
> modelo = keras.Sequential([
>     keras.Input(shape=(784,)),
>     keras.layers.Dense(30, activation="tanh"),
>     keras.layers.Dense(10),
> ])
> ```
>
> O `Dense` do Keras é a `Linear` daqui com a ativação embutida como argumento, à moda do `MLPClassifier`. O `keras.Input` da primeira linha não é camada nenhuma: ele só declara a forma da entrada, coisa que a nossa `Linear(784, 30)` faz no primeiro argumento. Fora isso, é a mesma lista de camadas com o mesmo nome de contêiner. E é do Keras que o desenho da biblioteca do livro-texto é vagamente copiado, como a seção de leituras adicionais do capítulo registra — o que torna esta a comparação mais próxima do capítulo inteiro.
>
> Duas diferenças que valem a pena.
>
> **Lá, `Sequential` é o caso simples, não o caso geral.** Redes de verdade raramente são uma sequência: há conexões que saltam camadas, entradas que se juntam no meio, saídas múltiplas. Para isso, o PyTorch pede que você escreva uma subclasse de `nn.Module` com o `forward` que quiser, e o autograd cuida do resto. O nosso `Sequential` é o único combinador que temos, e ele só sabe encadear.
>
> **O registro dos parâmetros é automático.** Atribuir `self.camada = nn.Linear(...)` dentro de um `nn.Module` registra aqueles parâmetros na hora — inclusive detectando o compartilhamento que a caixa acima quebrou. O nosso `params()` é um gerador escrito à mão, e ele devolve o que a lista contiver, repetido ou não.
>
> O que **não** muda é a leitura do objeto: quando você vê `Sequential([Linear(784, 30), Tanh(), Linear(30, 10)])` num código qualquer, o que está do outro lado é um `forward` que é um laço e um `backward` que é o mesmo laço com `reversed`.

## Perda e Otimização

> **📌 Nota**
>
> Esta seção corresponde a *Loss and Optimization* e a *Example: XOR Revisited*, do capítulo 19 de Grus (2019).

A rede está montada e sabe propagar gradientes. Falta o que fica **fora** dela: quem mede o erro e quem dá o passo. Nos capítulos anteriores, essas duas coisas apareciam escritas dentro do laço de treino, misturadas com tudo o mais. Aqui elas viram duas abstrações, pelo mesmo motivo de sempre: para poder trocá-las sem reescrever o resto.

### A perda

In [ ]:
import random
from typing import List, Iterable
from scratch.deep_learning import (Layer, Linear, Sequential, Sigmoid, Tensor,
                                   tensor_combine, tensor_sum, zeros_like, shape)

In [ ]:
class Loss:
    def loss(self, predicted: Tensor, actual: Tensor) -> float:
        """Quão boas são as nossas previsões? (Números maiores são piores.)"""
        raise NotImplementedError

    def gradient(self, predicted: Tensor, actual: Tensor) -> Tensor:
        """Como a perda muda quando as previsões mudam?"""
        raise NotImplementedError

Uma perda é um par: um número que diz o quanto erramos, e o gradiente desse número em relação às previsões. Repare que a segunda parte é exatamente o que a rede espera receber em `backward` — a perda é a fonte do gradiente que atravessa a rede inteira de trás para a frente.

A soma dos erros ao quadrado já apareceu tantas vezes neste livro que a implementação não guarda surpresa; o único cuidado é usar `tensor_combine`, para que ela funcione com tensores de qualquer forma:

In [ ]:
class SSE(Loss):
    """Função de perda que calcula a soma dos erros ao quadrado."""
    def loss(self, predicted: Tensor, actual: Tensor) -> float:
        # Calcula o tensor das diferenças ao quadrado
        squared_errors = tensor_combine(
            lambda predicted, actual: (predicted - actual) ** 2,
            predicted,
            actual)

        # E soma tudo
        return tensor_sum(squared_errors)

    def gradient(self, predicted: Tensor, actual: Tensor) -> Tensor:
        return tensor_combine(
            lambda predicted, actual: 2 * (predicted - actual),
            predicted,
            actual)

sse = SSE()
assert sse.loss([1, 2, 3], [10, 20, 30]) == 9 ** 2 + 18 ** 2 + 27 ** 2
assert sse.gradient([1, 2, 3], [10, 20, 30]) == [-18, -36, -54]

> **📌 Nota — O fator 2 voltou**
>
> A [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) tem uma caixa de aviso sobre um 2 que faltava: a derivada de $(o - t)^2$ é $2(o - t)$, e o código de lá calculava `output - target`, sem o 2. O efeito prático era só dobrar ou não a taxa de aprendizado, mas o comentário do código dizia "o gradiente da perda quadrática" quando era o gradiente da **metade** dela.
>
> Aqui o `2` está escrito. `SSE.loss` e `SSE.gradient` são consistentes uma com a outra, e o número que se acompanha na tela é o mesmo cujo gradiente está sendo descido. É uma correção pequena, e ela só ficou fácil de fazer porque a perda virou uma coisa nomeada: quando o cálculo da perda e o do gradiente moram no mesmo objeto, a inconsistência entre os dois fica visível.

### O otimizador

Até aqui, todo gradiente descendente do livro foi feito à mão, com uma linha assim:

```python
theta = gradient_step(theta, grad, -learning_rate)
```

É o `gradient_step` do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), a função de cinco linhas que a regressão linear, a múltipla, a logística e a rede do capítulo anterior chamaram sem exceção: ela recebe **um** vetor de parâmetros, **um** gradiente do mesmo tamanho, e devolve o vetor andado.

Isso não serve mais, por duas razões. A primeira é que uma rede tem **muitos** tensores de parâmetros, de formas diferentes, e todos precisam ser atualizados. A segunda é que queremos poder usar variantes mais espertas do gradiente descendente sem reescrever nada.

In [ ]:
class Optimizer:
    """
    Um otimizador atualiza os pesos de uma camada (no lugar) usando
    informação conhecida pela camada, pelo otimizador, ou por ambos.
    """
    def step(self, layer: Layer) -> None:
        raise NotImplementedError

class GradientDescent(Optimizer):
    def __init__(self, learning_rate: float = 0.1) -> None:
        self.lr = learning_rate

    def step(self, layer: Layer) -> None:
        for param, grad in zip(layer.params(), layer.grads()):
            # Atualiza param com um passo de gradiente
            param[:] = tensor_combine(
                lambda param, grad: param - grad * self.lr,
                param,
                grad)

O `zip(layer.params(), layer.grads())` é onde a correspondência posicional da [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-a-abstracao-de-camada.html) é usada. O otimizador não sabe o que aquele tensor é — se é uma matriz de pesos, um vetor de vieses ou outra coisa qualquer. Ele sabe que existe um gradiente na mesma posição, e isso basta.

> **⚠️ Atenção — `param[:] =`, e não `param =`**
>
> Aquela atribuição com fatia não é estilo. Se o código fosse `param = tensor_combine(...)`, ele redefiniria a variável local `param` e **não** afetaria o tensor guardado dentro da camada. O otimizador rodaria, não daria erro nenhum, e a rede jamais aprenderia.
>
> O livro-texto demonstra a diferença com um exemplo mínimo, e vale executá-lo:

In [ ]:
tensor = [[1, 2], [3, 4]]

for row in tensor:
    row = [0, 0]
assert tensor == [[1, 2], [3, 4]], "atribuição não atualiza a lista"

for row in tensor:
    row[:] = [0, 0]
assert tensor == [[0, 0], [0, 0]], "mas atribuição de fatia atualiza"

tensor

> Os dois `assert` passam. O primeiro laço rebatiza o nome `row` a cada volta e descarta o valor anterior; o segundo escreve **dentro** do objeto para o qual `row` aponta. Se isso surpreende, medite sobre o exemplo até fazer sentido — o resto do capítulo depende dele o tempo todo, e é um erro que não produz mensagem nenhuma.

Para mostrar que a abstração paga, o livro-texto implementa um segundo otimizador. A ideia do **momento** é não reagir demais a cada gradiente novo: em vez disso, mantemos uma média móvel dos gradientes vistos, atualizamos essa média a cada passo e andamos na direção dela.

In [ ]:
class Momentum(Optimizer):
    def __init__(self, learning_rate: float, momentum: float = 0.9) -> None:
        self.lr = learning_rate
        self.mo = momentum
        self.updates: List[Tensor] = []      # média móvel

    def step(self, layer: Layer) -> None:
        # Se não há atualizações anteriores, começa com zeros.
        if not self.updates:
            self.updates = [zeros_like(grad) for grad in layer.grads()]

        for update, param, grad in zip(self.updates,
                                       layer.params(),
                                       layer.grads()):
            # Aplica o momento
            update[:] = tensor_combine(
                lambda u, g: self.mo * u + (1 - self.mo) * g,
                update,
                grad)

            # E então dá um passo de gradiente
            param[:] = tensor_combine(
                lambda p, u: p - self.lr * u,
                param,
                update)

> **🔷 Conceito**
>
> `Momentum` guarda estado entre chamadas — a média móvel `self.updates` — e `GradientDescent` não guarda nada. Do ponto de vista de quem usa, isso é invisível: os dois têm um `step(layer)` e nada mais.
>
> Essa é a diferença entre uma função e uma abstração. O `gradient_step` do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) era uma função pura — sem memória de um passo para o outro —, e trocar a regra de atualização significava trocar a função e todo o código que a chamava. Um `Momentum` escrito como função pura nem seria possível: a média móvel precisa sobreviver entre as chamadas, e a única forma de guardá-la seria passá-la para dentro e para fora da função, mudando a assinatura de todo mundo. Um `Optimizer` é um objeto com estado próprio, e trocar a regra é trocar **uma linha**: a que constrói o objeto.

### Exemplo: XOR revisitado

Vamos ver o quanto ficou fácil treinar uma rede que calcula o XOR. Os dados são os quatro exemplos de sempre — o problema inteiro:

In [ ]:
xs = [[0., 0], [0., 1], [1., 0], [1., 1]]
ys = [[0.], [1.], [1.], [0.]]

E a rede se monta em cinco linhas. Repare que agora dá para **deixar de fora** a sigmoide final, coisa que a representação do capítulo anterior não permitia — lá a sigmoide estava soldada dentro de `neuron_output`, e todo neurônio de toda camada tinha uma:

In [ ]:
random.seed(0)

net = Sequential([
    Linear(input_dim=2, output_dim=2),
    Sigmoid(),
    Linear(input_dim=2, output_dim=1)
])

O laço de treino usa as abstrações de `Loss` e `Optimizer`, o que permite trocar qualquer uma das duas sem tocar no resto:

In [ ]:
import tqdm

optimizer = GradientDescent(learning_rate=0.1)
loss = SSE()
perdas_gd = []

with tqdm.trange(3000) as t:
    for epoch in t:
        epoch_loss = 0.0

        for x, y in zip(xs, ys):
            predicted = net.forward(x)
            epoch_loss += loss.loss(predicted, y)
            gradient = loss.gradient(predicted, y)
            net.backward(gradient)

            optimizer.step(net)

        perdas_gd.append(epoch_loss)
        t.set_description(f"xor loss {epoch_loss:.3f}")

print(f"perda final: {epoch_loss:.8f}")

> **🟩 Exemplo — Os dois laços, lado a lado**
>
> Vale pôr o laço acima ao lado do da [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html), que resolvia o mesmo problema:
>
> ```python
> # Capítulo 15
> for epoch in range(20000):
>     for x, y in zip(xs, ys):
>         gradients = sqerror_gradients(network, x, y)
>         network = [[gradient_step(neuron, grad, -learning_rate)
>                     for neuron, grad in zip(layer, layer_grad)]
>                    for layer, layer_grad in zip(network, gradients)]
> ```
>
> ```python
> # Capítulo 16
> for epoch in range(3000):
>     for x, y in zip(xs, ys):
>         predicted = net.forward(x)
>         gradient = loss.gradient(predicted, y)
>         net.backward(gradient)
>         optimizer.step(net)
> ```
>
> Os dois cabem em meia dúzia de linhas, e o de cima até tem menos comandos. A diferença não é tamanho — é **o que cada comando sabe**.
>
> O laço de cima conhece a estrutura da rede: ele sabe que `network` é uma lista de camadas de neurônios, e reconstrói essa estrutura inteira a cada exemplo, com duas compreensões de lista aninhadas. Trocar a topologia quebra essas compreensões; trocar a ativação exige reescrever `sqerror_gradients`; trocar o otimizador exige trocar `gradient_step` por outra coisa dentro do aninhamento.
>
> O laço de baixo não conhece nada. `net` poderia ter três camadas ou trinta; `loss` poderia ser outra; `optimizer` poderia ser `Momentum`. **Nenhuma dessas trocas mexe numa linha deste laço** — e é por isso que o mesmo laço vai treinar a rede do Fizz Buzz na próxima seção e a rede de reconhecimento de dígitos na última, sem alteração.

Trocar o otimizador é uma linha. Vamos aproveitar para medir se o momento ajuda:

In [ ]:
random.seed(0)

net_mo = Sequential([
    Linear(input_dim=2, output_dim=2),
    Sigmoid(),
    Linear(input_dim=2, output_dim=1)
])

optimizer_mo = Momentum(learning_rate=0.1, momentum=0.9)     # <- a única troca
perdas_mo = []

with tqdm.trange(3000) as t:
    for epoch in t:
        epoch_loss = 0.0
        for x, y in zip(xs, ys):
            predicted = net_mo.forward(x)
            epoch_loss += loss.loss(predicted, y)
            net_mo.backward(loss.gradient(predicted, y))
            optimizer_mo.step(net_mo)
        perdas_mo.append(epoch_loss)
        t.set_description(f"xor loss {epoch_loss:.3f}")

for nome, perdas in [("GradientDescent", perdas_gd), ("Momentum", perdas_mo)]:
    primeira = next(i + 1 for i, v in enumerate(perdas) if v < 0.01)
    print(f"{nome:16s} primeiro epoch com perda < 0,01: {primeira}")

In [ ]:
# Figura: Perda por epoch no XOR, com dois otimizadores diferentes e a mesma inicialização. O eixo horizontal está cortado em 1.600 epochs: daí em diante as duas curvas seguem coladas em zero até o epoch 3.000.
from matplotlib import pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(range(1, 3001), perdas_gd, label="GradientDescent (taxa 0,1)")
plt.plot(range(1, 3001), perdas_mo, label="Momentum (taxa 0,1; momento 0,9)")
plt.xlim(0, 1600)
plt.xlabel("epoch")
plt.ylabel("soma dos erros ao quadrado")
plt.title("Treino do XOR")
plt.legend()
plt.show()

As duas curvas contam a mesma história em ritmos diferentes: uma queda vertical no primeiro epoch, um trecho longo e quase plano, e só então a descida até zero. A curva azul ainda faz uma **parada intermediária** por volta de 0,6, entre os epochs 1.000 e 1.150, que a laranja não faz — o momento atravessa aquele trecho sem se deter nele.

Os patamares são o mesmo fenômeno que a [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) documentou no Fizz Buzz: a rede fica presa numa região de gradiente minúsculo antes de escapar. O momento não elimina o platô; ele o encurta, e é exatamente para isso que ele existe — a média móvel acumula empurrões pequenos e repetidos na mesma direção até que a soma deles seja grande o bastante para sair dali.

### O que a rede aprendeu, desta vez

In [ ]:
w1, b1, w2, b2 = list(net.params())

print("camada 1, pesos: ", [[round(v, 4) for v in linha] for linha in w1])
print("camada 1, vieses:", [round(v, 4) for v in b1])
print("camada 2, pesos: ", [[round(v, 4) for v in linha] for linha in w2])
print("camada 2, viés:  ", [round(v, 4) for v in b2])

In [ ]:
for x in xs:
    saida = net.forward(x)
    escondida = net.layers[1].sigmoids
    print(f"{x} -> escondida [{escondida[0]:.4f}, {escondida[1]:.4f}]"
          f"   saída {saida[0]:.4f}")

> **🔷 Conceito — Três treinos, três soluções diferentes, a mesma função**
>
> Olhe os pesos da primeira camada: **todos negativos**. Os dois neurônios escondidos são funções decrescentes das duas entradas — os dois calculam alguma versão de "nem uma nem outra". O que os distingue é a inclinação: o segundo tem pesos por volta de −4 e −3,4 e desaba para quase zero assim que qualquer entrada liga; o primeiro, com −1,6 e −1,5, decai devagar. E a camada de saída faz a diferença entre os dois, com pesos $+3{,}2$ e $-3{,}5$.
>
> O resultado é um detector de "exatamente uma entrada ligada", construído a partir da **distância entre dois "nem-nem" de dureza diferente**. Não é o "ou, mas não e" que a [seção 15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-retropropagacao.html) encontrou, e também não é o "nem-nem, e, nem-nem" que o livro-texto relata para o experimento dele.
>
> Três treinos, três soluções internas diferentes, todas calculando o XOR corretamente. Isso confirma, agora com uma terceira amostra, o que o capítulo anterior já tinha dito: **a rede tem várias soluções equivalentes, e a que ela encontra depende de onde a inicialização a colocou.** A diferença aqui é que a inicialização mudou não só de semente, mas de esquema — `random_tensor(init='xavier')` em vez de `random.random()` — e a arquitetura perdeu a sigmoide final.
>
> Repare, aliás, que as saídas agora batem em 1,0000 e 0,0000 — e que a primeira linha imprime `-0.0000`, um zero negativo de ponto flutuante, o que só acontece porque o valor é minúsculo **e** negativo. Sem a sigmoide final, a saída da rede não está mais presa em $[0, 1]$: ela pode passar do alvo, e a perda quadrática pode de fato chegar a zero, coisa que uma sigmoide nunca permitiria.

> **💡 Dica — Na prática: `torch.optim` e as funções de perda prontas**
>
> As duas abstrações desta seção existem, com estes nomes, em qualquer framework:
>
> ```python
> import torch.nn as nn
> import torch.optim as optim
>
> perda = nn.MSELoss()
> otimizador = optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9)
> #                                           ^^^^^^^ não é 0.1: veja abaixo
> ```
>
> Sem o argumento `momentum`, o `optim.SGD` é exatamente o nosso `GradientDescent`. Com `momentum=0.9` ele é o nosso `Momentum` **na ideia, não na convenção** — e a diferença tem tamanho.
>
> O nosso acumula uma **média móvel** do gradiente, $u \leftarrow \mu u + (1 - \mu) g$: os pesos $\mu$ e $1-\mu$ somam 1, então, diante de um gradiente constante $g$, o `update` converge para o próprio $g$, e o passo `lr * u` nunca passa de um passo de gradiente comum. O PyTorch, com o padrão `dampening=0`, acumula uma **soma amortecida**, $b \leftarrow \mu b + g$. Sem o fator $(1 - \mu)$, o buffer converge para $g/(1-\mu)$ — com $\mu = 0{,}9$, **dez vezes** $g$.
>
> Ou seja: para a mesma taxa de aprendizado, o passo efetivo do PyTorch é dez vezes o nosso, e traduzir um `Momentum(learning_rate=0.1, momentum=0.9)` daqui para lá pede `lr=0.01` — que é justamente o valor no bloco acima, e o motivo do comentário nele. É a mesma matemática com outra normalização — o tipo de detalhe que só aparece quando se põem as duas fórmulas lado a lado, e a razão pela qual "é o mesmo otimizador" nunca dispensa olhar a documentação.
>
> O laço de treino em PyTorch é este:
>
> ```python
> saida = modelo(x)
> erro = perda(saida, y)
> otimizador.zero_grad()
> erro.backward()
> otimizador.step()
> ```
>
> Cinco linhas, das quais quatro têm equivalente exato no nosso laço. A que não tem é o `zero_grad()`, e a razão dela é instrutiva: o PyTorch **acumula** gradientes em `.grad` em vez de sobrescrevê-los, justamente para que pesos compartilhados funcionem — o caso que a [seção 16.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/04-redes-como-sequencia-de-camadas.html) mostrou quebrado na nossa biblioteca. O preço é que você precisa zerá-los à mão a cada passo, e esquecer disso é um dos bugs mais comuns de quem começa.
>
> Sobre os otimizadores em si: `SGD` e `SGD` com momento são o começo de uma lista longa. O padrão de fato hoje é o **Adam**, que mantém médias móveis do gradiente **e** do gradiente ao quadrado, e usa a segunda para dar um passo de tamanho diferente em cada parâmetro. Ele não é conceitualmente distante do `Momentum` que você escreveu — é o mesmo truque de média móvel, aplicado duas vezes e combinado. O `MLPClassifier` do `scikit-learn`, aliás, usa Adam por padrão, o que explica em parte a tabela de resultados do callout final do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html).

## Outras Funções de Ativação

> **📌 Nota**
>
> Esta seção corresponde a *Other Activation Functions* e a *Example: FizzBuzz Revisited*, do capítulo 19 de Grus (2019).

A sigmoide caiu em desuso, e há dois motivos.

O primeiro é que $\sigma(0) = 1/2$: um neurônio cujas entradas somam zero produz saída **positiva**. Isso significa que a saída de uma camada inteira é sistematicamente deslocada para cima. (O livro-texto para aqui; a leitura usual da área, que ele não faz, é que uma ativação não centrada em zero atrapalha o treino porque a camada seguinte recebe entradas todas do mesmo sinal — e a `tanh`, logo abaixo, é a correção direta disso.)

O segundo já foi medido na [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-a-abstracao-de-camada.html): o gradiente da sigmoide é próximo de zero para entradas muito grandes e muito pequenas. A derivada em 3 vale 0,0452, contra 0,25 no pico. Um neurônio que caia nessa região fica **saturado** — o gradiente que passa por ele é minúsculo, os pesos dele quase não se movem, e ele pode ficar preso ali indefinidamente.

### A tangente hiperbólica

A substituta popular é a `tanh`, outra função em forma de S, que vai de −1 a 1 e devolve 0 quando a entrada é 0. A derivada dela é simplesmente $1 - \tanh(x)^2$, o que torna a camada fácil de escrever:

In [ ]:
import math
import random
from typing import List
from scratch.deep_learning import (Layer, Linear, Sequential, Sigmoid, Tensor,
                                   tensor_apply, tensor_combine, Momentum, SSE)
from scratch.neural_networks import sigmoid

In [ ]:
def tanh(x: float) -> float:
    # Se x é muito grande ou muito pequeno, tanh é (essencialmente) 1 ou -1.
    # Verificamos isso porque, por exemplo, math.exp(1000) levanta um erro.
    if x < -100:  return -1
    elif x > 100: return 1

    em2x = math.exp(-2 * x)
    return (1 - em2x) / (1 + em2x)

class Tanh(Layer):
    def forward(self, input: Tensor) -> Tensor:
        # Guarda a saída da tanh para usar na passada para trás.
        self.tanh = tensor_apply(tanh, input)
        return self.tanh

    def backward(self, gradient: Tensor) -> Tensor:
        return tensor_combine(
            lambda tanh, grad: (1 - tanh ** 2) * grad,
            self.tanh,
            gradient)

> **📌 Nota**
>
> As duas linhas de guarda no começo de `tanh` não são zelo excessivo: `math.exp(200)` estoura o alcance do `float` e levanta `OverflowError`. É o mesmo problema de saturação numérica que o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) documentou para a função logística, tratado aqui com um recorte explícito em vez de deixar estourar.
>
> Repare também que a camada guarda `self.tanh`, exatamente como a `Sigmoid` guardava `self.sigmoids`, e pelo mesmo motivo: a derivada se escreve em função da própria saída.

### A retificadora

Em redes maiores, a substituta popular é a **ReLU** (*rectified linear unit*), que vale 0 para entradas negativas e a identidade para entradas positivas:

In [ ]:
class Relu(Layer):
    def forward(self, input: Tensor) -> Tensor:
        self.input = input
        return tensor_apply(lambda x: max(x, 0), input)

    def backward(self, gradient: Tensor) -> Tensor:
        return tensor_combine(lambda x, grad: grad if x > 0 else 0,
                              self.input,
                              gradient)

Repare que o `backward` da ReLU não faz multiplicação nenhuma: ele **deixa o gradiente passar inteiro** onde a entrada era positiva, e o zera onde era negativa. É o que a derivada de $\max(x, 0)$ é — 1 ou 0 — e é a razão pela qual a ReLU não satura para entradas grandes: por maior que seja a pré-ativação, o gradiente que a atravessa continua sendo o mesmo que chegou.

Vale ver as três lado a lado:

In [ ]:
# Figura: Em cima, as três funções de ativação. Embaixo, as derivadas delas.
from matplotlib import pyplot as plt

grade = [-5 + i / 50 for i in range(501)]

fig, eixos = plt.subplots(2, 3, figsize=(10.5, 5.5), sharex=True)

def desenha(coluna, titulo, f, df, buraco=False):
    cima, baixo = eixos[0][coluna], eixos[1][coluna]
    cima.plot(grade, [f(x) for x in grade], color='C0')
    cima.set_title(titulo)
    cima.set_ylim(-1.3, 2.3)
    cima.axhline(0, color='0.8', linewidth=0.8)
    cima.axvline(0, color='0.8', linewidth=0.8)
    if buraco:
        baixo.plot([x for x in grade if x < 0], [0.0 for x in grade if x < 0], color='C1')
        baixo.plot([x for x in grade if x > 0], [1.0 for x in grade if x > 0], color='C1')
        baixo.scatter([0], [0], s=45, facecolors='white', edgecolors='C1',
                      zorder=3, linewidths=1.4)
    else:
        baixo.plot(grade, [df(x) for x in grade], color='C1')
    baixo.set_ylim(-0.15, 1.25)
    baixo.axhline(0, color='0.8', linewidth=0.8)
    baixo.axvline(0, color='0.8', linewidth=0.8)
    baixo.set_xlabel("x")

desenha(0, "sigmoide", sigmoid, lambda x: sigmoid(x) * (1 - sigmoid(x)))
desenha(1, "tanh", tanh, lambda x: 1 - tanh(x) ** 2)
desenha(2, "ReLU", lambda x: max(x, 0.0), None, buraco=True)

eixos[0][0].set_ylabel("f(x)")
eixos[1][0].set_ylabel("f'(x)")
plt.tight_layout()
plt.show()

A linha de baixo é a que decide. A derivada da sigmoide chega no máximo a 0,25 e desaba dos dois lados; a da `tanh` chega a 1 e desaba dos dois lados; a da ReLU vale 1 em todo o lado positivo, sem desabar nunca — e vale 0 em todo o lado negativo, o que é o defeito dela: um neurônio ReLU que caia na região negativa e fique lá não recebe gradiente algum e **morre**.

Repare também no círculo vazado em $x = 0$ no gráfico da derivada da ReLU. A função não é derivável ali, e o código resolve isso por decreto: `grad if x > 0 else 0` escolhe 0. Na prática funciona, porque a chance de uma pré-ativação cair exatamente em zero é desprezível.

### Exemplo: Fizz Buzz revisitado

Vamos usar a biblioteca para refazer o problema da [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) — prever, a partir da representação binária de um número, se ele é múltiplo de 3, de 5, de 15 ou de nenhum dos dois. **O ponto não é que funciona de novo; é o que sumiu do código.**

Os dados são idênticos, e as funções de codificação vêm do pacote, escritas naquele capítulo:

In [ ]:
from scratch.neural_networks import binary_encode, fizz_buzz_encode, argmax

xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

len(xs), len(xs[0]), len(ys[0])

923 exemplos de treino, dez bits de entrada, quatro saídas. Os números de 1 a 100 ficam de fora, para servir de teste.

> **🟩 Exemplo — As duas redes, lado a lado**
>
> ```python
> # Capítulo 15
> NUM_HIDDEN = 25
> network = [
>     [[random.random() for _ in range(10 + 1)] for _ in range(NUM_HIDDEN)],
>     [[random.random() for _ in range(NUM_HIDDEN + 1)] for _ in range(4)]
> ]
> ```
>
> ```python
> # Capítulo 16
> NUM_HIDDEN = 25
> net = Sequential([
>     Linear(input_dim=10, output_dim=NUM_HIDDEN, init='uniform'),
>     Tanh(),
>     Linear(input_dim=NUM_HIDDEN, output_dim=4, init='uniform'),
>     Sigmoid()
> ])
> ```
>
> O de cima diz **onde os números moram**: uma lista de 25 listas de 11 floats, e uma lista de 4 listas de 26 floats. O `+ 1` em cada comprimento é o viés grudado no fim, e quem lê precisa saber disso para entender a expressão.
>
> O de baixo diz **o que a rede faz**: recebe 10, produz 25, aplica `tanh`, produz 4, aplica sigmoide. As dimensões estão escritas com os nomes delas; o viés não aparece porque virou responsabilidade da camada.
>
> E há uma linha no de baixo que **não tem equivalente possível** no de cima: `Tanh()`. No capítulo anterior, a sigmoide estava soldada dentro de `neuron_output`, e a derivada dela aparecia escrita à mão dentro de `sqerror_gradients`. Usar outra ativação exigiria editar as duas funções. Aqui é um objeto na lista.

In [ ]:
random.seed(0)

NUM_HIDDEN = 25

net = Sequential([
    Linear(input_dim=10, output_dim=NUM_HIDDEN, init='uniform'),
    Tanh(),
    Linear(input_dim=NUM_HIDDEN, output_dim=4, init='uniform'),
    Sigmoid()
])

def fizzbuzz_accuracy(low: int, hi: int, net: Layer) -> float:
    num_correct = 0
    for n in range(low, hi):
        x = binary_encode(n)
        predicted = argmax(net.forward(x))
        actual = argmax(fizz_buzz_encode(n))
        if predicted == actual:
            num_correct += 1

    return num_correct / (hi - low)

O laço de treino é o **mesmo** da seção anterior. Nem uma linha muda: `net`, `loss` e `optimizer` mudaram de conteúdo, e o laço não sabe disso.

In [ ]:
import time
import tqdm

optimizer = Momentum(learning_rate=0.1, momentum=0.9)
loss = SSE()

perdas = []
marcos = [1, 25, 50, 100, 200, 300]
historico = {}

t0 = time.time()

with tqdm.trange(300) as t:
    for epoch in t:
        epoch_loss = 0.0

        for x, y in zip(xs, ys):
            predicted = net.forward(x)
            epoch_loss += loss.loss(predicted, y)
            gradient = loss.gradient(predicted, y)
            net.backward(gradient)

            optimizer.step(net)

        perdas.append(epoch_loss)
        t.set_description(f"fb loss: {epoch_loss:.2f}")

        if epoch + 1 in marcos:
            historico[epoch + 1] = (epoch_loss,
                                    fizzbuzz_accuracy(101, 1024, net),
                                    fizzbuzz_accuracy(1, 101, net))

In [ ]:
segundos = time.time() - t0
print(f"300 epochs em {segundos:.1f} s ({segundos / 300:.3f} s por epoch)\n")

print("epoch      perda   treino   teste")
for epoch in marcos:
    perda, treino, teste = historico[epoch]
    print(f"{epoch:5d}  {perda:9.2f}   {treino:.3f}   {teste:.3f}")

In [ ]:
# Figura: Soma dos erros ao quadrado sobre os 923 exemplos de treino, epoch a epoch — desta vez com `Tanh`, `SSE` e `Momentum`, em 300 epochs. A curva sai de um patamar em torno de 2.769 e desce em degraus, não continuamente.
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(perdas) + 1), perdas)
plt.xlabel("epoch")
plt.ylabel("soma dos erros ao quadrado")
plt.title("Fizz Buzz com Tanh, SSE e Momentum")
plt.show()

A curva é uma **escada**. Ela fica exatamente parada em 2.769 por uns dezessete epochs, despenca para 1.966, fica parada de novo por uns quarenta, cai em três degraus seguidos entre os epochs 57 e 72, e só a partir dali começa a descer de forma contínua.

Os patamares são o mesmo fenômeno que a [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) documentou, e a lição de tê-los de novo é esta: trocamos a ativação, o otimizador, a inicialização e a representação inteira da rede, e o platô não foi embora. Ele não é um artefato do código; é uma propriedade do problema.

O que mudou foi o **conteúdo** do platô. Lá, a rede presa respondia sempre a classe mais frequente e acertava 53,4% — o palpite preguiçoso do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html). Aqui, olhe a coluna de treino da tabela nos dois primeiros patamares: 13,3% e 13,5%, **abaixo** daquele palpite. A rede não está presa na resposta mais comum; está presa numa resposta pior que ela.

> **❗ Importante — Por que 300 epochs, e não 1.000**
>
> O livro-texto roda **1.000** epochs neste exemplo e relata 90% de acerto no teste, com a observação de que treinar mais melhoraria ainda.
>
> Aqui rodamos 300. O motivo está impresso acima: um epoch custa cerca de 0,12 segundo nesta biblioteca, então 1.000 epochs custariam **dois minutos** de renderização — e este livro é renderizado por inteiro, do zero, a cada publicação, com o cache de execução apagado de propósito para provar que nada vem da rede. Trezentos epochs custam menos de um terço disso e chegam perto o suficiente do resultado do livro-texto para o argumento ficar de pé.
>
> Esse tipo de corte vai reaparecer, muito maior, na [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html). Ele é a razão pela qual o custo de treino é um assunto deste capítulo e não um detalhe de bastidor.

> **💡 Dica — Na prática: a ativação como objeto, e a lista que não acaba**
>
> No PyTorch, cada ativação é uma camada, exatamente como aqui:
>
> ```python
> import torch.nn as nn
>
> modelo = nn.Sequential(
>     nn.Linear(10, 25),
>     nn.Tanh(),          # ou nn.ReLU(), nn.GELU(), nn.SiLU(), nn.LeakyReLU()...
>     nn.Linear(25, 4),
> )
> ```
>
> Trocar `nn.Tanh()` por `nn.ReLU()` é editar uma palavra — e no nosso código também é, o que é justamente o resultado desta seção. No `MLPClassifier` do `scikit-learn`, ao contrário, a ativação é uma **string** passada ao construtor (`activation='relu'`), escolhida entre quatro valores permitidos; não há como usar uma quinta.
>
> A lista de ativações usadas hoje é longa e continua crescendo. Duas que valem citar, porque nasceram dos defeitos que esta seção mostrou:
>
> - A **Leaky ReLU** troca o zero do lado negativo por uma reta de inclinação pequena, para que um neurônio que caia lá continue recebendo algum gradiente. É a correção direta do problema do neurônio morto.
> - A **GELU**, hoje padrão nos modelos de linguagem, é uma versão suave da ReLU — sem o bico em zero, e portanto derivável em toda parte.
>
> Repare no padrão: as ativações modernas são todas variações sobre o mesmo eixo, o de manter um gradiente utilizável passando. É o mesmo critério que fez a sigmoide ser aposentada, aplicado mais vezes.

## Softmax, Entropia Cruzada e Dropout

> **📌 Nota**
>
> Esta seção corresponde a *Softmaxes and Cross-Entropy* e a *Dropout*, do capítulo 19 de Grus (2019).

A rede da seção anterior termina numa camada `Sigmoid`, o que significa que a saída dela é um vetor de números entre 0 e 1. Em particular, ela pode devolver um vetor só de zeros, ou um vetor só de uns.

Num problema de classificação, o que queremos é outra coisa: um 1 na classe certa e 0 em todas as outras. As previsões nunca serão tão perfeitas, mas gostaríamos que fossem, pelo menos, uma **distribuição de probabilidade** sobre as classes. Se um modelo com duas classes devolve $[0, 0]$, é difícil dar sentido a isso — ele acha que a entrada não pertence a classe nenhuma? Já $[0{,}4;\ 0{,}6]$ se lê sem esforço: 40% de chance de ser a primeira classe, 60% de ser a segunda.

> **❗ Importante — A dívida da seção 15.4**
>
> O capítulo anterior fechou com exatamente essa observação, e a deixou pendurada. Lá, a rede do Fizz Buzz devolvia quatro sigmoides independentes, e a caixa "Estes quatro números não são uma distribuição de probabilidade" mostrou uma saída cuja soma dava 1,0769 e outra cuja soma dava 0,9062.
>
> Cada neurônio de saída tinha a sua própria sigmoide e produzia um número em $[0, 1]$ **independentemente dos outros**. Quatro números em $[0, 1]$ não formam uma distribuição só porque estão lado a lado, e ler o maior deles como "confiança" era inventar uma garantia que o modelo não dava.
>
> Esta seção paga essa dívida.

### A softmax

A correção é abandonar a camada `Sigmoid` final e usar a função **softmax**, que converte um vetor de números reais num vetor de probabilidades. Calculamos $e^x$ para cada número — o que produz um vetor de positivos — e dividimos cada um pela soma. O resultado é um punhado de números positivos que somam 1.

Só há um cuidado: $e^{1000}$ estoura em Python. Antes de exponenciar, subtraímos o maior valor do vetor, o que não muda as probabilidades e é mais seguro de calcular:

In [ ]:
import math
import random
from typing import List
from scratch.deep_learning import (Layer, Linear, Sequential, Tanh, Tensor,
                                   is_1d, tensor_apply, tensor_combine, tensor_sum,
                                   Loss, Momentum)
from scratch.neural_networks import sigmoid
import operator

In [ ]:
def softmax(tensor: Tensor) -> Tensor:
    """Softmax ao longo da última dimensão"""
    if is_1d(tensor):
        # Subtrai o maior valor, por estabilidade numérica.
        largest = max(tensor)
        exps = [math.exp(x - largest) for x in tensor]

        sum_of_exps = sum(exps)                 # este é o "peso" total
        return [exp_i / sum_of_exps             # a probabilidade é a fração
                for exp_i in exps]              # do peso total
    else:
        return [softmax(tensor_i) for tensor_i in tensor]

Vale ver os dois efeitos lado a lado — o de normalizar e o de ser insensível a deslocamentos:

In [ ]:
v = [2.0, 1.0, 0.1, -3.0]

sigmoides = [sigmoid(x) for x in v]
probs = softmax(v)

print("vetor            ", [round(x, 4) for x in v])
print("sigmoides        ", [round(x, 4) for x in sigmoides],
      " soma:", round(sum(sigmoides), 4))
print("softmax          ", [round(x, 4) for x in probs],
      " soma:", round(sum(probs), 4))
print("softmax(v + 100) ", [round(x, 4) for x in softmax([x + 100 for x in v])])

As quatro sigmoides somam 2,1843, que não é nada. A softmax soma 1, e continua somando o mesmo depois de somar 100 a todo mundo — o que é a justificativa do truque de subtrair o máximo: deslocar o vetor inteiro não altera o resultado, então podemos deslocá-lo para onde a exponencial não estoura.

> **🔷 Conceito — A softmax é a logística, para mais de duas classes**
>
> O [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) construiu a função logística para espremer uma combinação linear dentro de $[0, 1]$ e poder lê-la como probabilidade. A softmax faz o mesmo trabalho quando há mais de duas classes — e, com exatamente duas, **ela é a logística**:

In [ ]:
for z in (-2.0, 0.0, 1.5):
    print(f"z = {z:5.1f}   softmax([z, 0]) = "
          f"{[round(p, 6) for p in softmax([z, 0.0])]}"
          f"   logística(z) = {sigmoid(z):.6f}")

> A primeira componente da softmax de $[z, 0]$ é exatamente $\sigma(z)$, e a segunda é $1 - \sigma(z)$. A conta é de uma linha: $e^z / (e^z + e^0) = 1/(1 + e^{-z})$.
>
> Isso arruma os dois capítulos numa hierarquia só. A regressão logística prevê uma probabilidade entre duas classes; a softmax prevê uma distribuição entre $k$; e a primeira é o caso $k = 2$ da segunda. É por isso que a última seção deste capítulo vai chamar `Linear(784, 10)` seguido de softmax de "regressão logística multiclasse" sem nenhuma licença poética.

### A entropia cruzada

Uma vez que a rede produz probabilidades, usa-se uma perda diferente: a **entropia cruzada**, também chamada de log-verossimilhança negativa.

O nome já entrega o argumento, e ele é o do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html): lá, o uso dos mínimos quadrados na regressão linear foi justificado mostrando que, sob certas hipóteses, os coeficientes de mínimos quadrados **maximizam a verossimilhança** dos dados observados. Aqui a coisa é análoga: se as saídas da rede são probabilidades, a entropia cruzada é a log-verossimilhança negativa dos dados observados — e minimizá-la é maximizar a verossimilhança.

In [ ]:
class SoftmaxCrossEntropy(Loss):
    """
    Esta é a log-verossimilhança negativa dos valores observados, dado o
    modelo de rede neural. Então, se escolhemos pesos que a minimizam,
    o nosso modelo estará maximizando a verossimilhança dos dados observados.
    """
    def loss(self, predicted: Tensor, actual: Tensor) -> float:
        # Aplica a softmax para obter probabilidades
        probabilities = softmax(predicted)

        # Isto vale log p_i para a classe verdadeira i, e 0 para as demais.
        # Somamos um tantinho a p para evitar log(0).
        likelihoods = tensor_combine(lambda p, act: math.log(p + 1e-30) * act,
                                     probabilities,
                                     actual)

        # E então somamos os negativos.
        return -tensor_sum(likelihoods)

    def gradient(self, predicted: Tensor, actual: Tensor) -> Tensor:
        probabilities = softmax(predicted)

        # Que equação agradável, não?
        return tensor_combine(lambda p, actual: p - actual,
                              probabilities,
                              actual)

> **❗ Importante — A softmax não entra na rede**
>
> Repare no que acabou de acontecer: a softmax está dentro da **perda**, não da rede. A rede termina numa `Linear` e devolve números reais quaisquer, sem restrição de sinal ou de escala.
>
> Isso não é arrumação estética. Olhe o `gradient`: ele é `probabilities - actual`, e nada mais. Duas subtrações e uma exponencial. Se a softmax fosse uma camada da rede, o gradiente teria de atravessá-la — e a derivada da softmax em relação às suas entradas é uma matriz, não um vetor, porque cada saída depende de **todas** as entradas.
>
> Quando a softmax e a entropia cruzada são derivadas **juntas**, essa matriz se cancela contra a derivada do logaritmo e sobra a subtração. É por isso que praticamente toda biblioteca oferece uma perda chamada `CrossEntropyLoss` que aplica a softmax internamente, e por isso que aplicar softmax na rede **e** usar essa perda é um dos erros mais comuns de quem começa: a softmax acaba aplicada duas vezes, o gradiente fica achatado e o treino arrasta.

### Fizz Buzz, mais uma vez

O livro-texto relata que treinar a mesma rede do Fizz Buzz com `SoftmaxCrossEntropy` costuma ser **muito** mais rápido — em muito menos epochs. A explicação dele vale ser lida com atenção, porque é geométrica.

Para prever a classe 0 no arranjo anterior — linear seguido de sigmoide —, a rede precisa que a primeira saída seja um número **grande e positivo** e que as outras três sejam **grandes e negativas**. Com a softmax, basta que a primeira saída seja **maior** que as outras três. Há muito mais jeitos de a segunda condição acontecer do que a primeira, e é razoável esperar que seja mais fácil encontrar pesos que a satisfaçam.

A rede é a mesma da seção anterior, com uma camada a menos:

In [ ]:
from scratch.neural_networks import binary_encode, fizz_buzz_encode, argmax

xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

NUM_HIDDEN = 25

random.seed(0)

net = Sequential([
    Linear(input_dim=10, output_dim=NUM_HIDDEN, init='uniform'),
    Tanh(),
    Linear(input_dim=NUM_HIDDEN, output_dim=4, init='uniform')
    # sem camada de sigmoide no fim agora
])

def fizzbuzz_accuracy(low: int, hi: int, net: Layer) -> float:
    num_correct = 0
    for n in range(low, hi):
        if argmax(net.forward(binary_encode(n))) == argmax(fizz_buzz_encode(n)):
            num_correct += 1
    return num_correct / (hi - low)

E o laço de treino é, de novo, **o mesmo**. Só a perda mudou:

In [ ]:
import time
import tqdm

optimizer = Momentum(learning_rate=0.1, momentum=0.9)
loss = SoftmaxCrossEntropy()                       # <- a única troca

marcos = [1, 10, 25, 50, 75, 100]
historico = {}

t0 = time.time()

with tqdm.trange(100) as t:
    for epoch in t:
        epoch_loss = 0.0

        for x, y in zip(xs, ys):
            predicted = net.forward(x)
            epoch_loss += loss.loss(predicted, y)
            gradient = loss.gradient(predicted, y)
            net.backward(gradient)

            optimizer.step(net)

        t.set_description(f"fb loss: {epoch_loss:.3f}")

        if epoch + 1 in marcos:
            historico[epoch + 1] = (epoch_loss,
                                    fizzbuzz_accuracy(101, 1024, net),
                                    fizzbuzz_accuracy(1, 101, net))

In [ ]:
segundos = time.time() - t0
print(f"100 epochs em {segundos:.1f} s ({segundos / 100:.3f} s por epoch)\n")

print("epoch      perda   treino   teste")
for epoch in marcos:
    perda, treino, teste = historico[epoch]
    print(f"{epoch:5d}  {perda:9.3f}   {treino:.3f}   {teste:.3f}")

A coluna do teste desmente a pressa: ela vai de 0,530 para 0,460 e para **0,340** no epoch 25 — a acurácia **piora durante um quarto do treino** — antes de disparar para 0,730, 0,820 e 0,970, enquanto a perda cai sem interrupção o tempo todo. As duas leituras não se contradizem: a rede passa esses primeiros epochs reorganizando as quatro fronteiras de decisão ao mesmo tempo, e uma fronteira já movida mas ainda não chegada acerta menos que a inicial, ainda que a perda — que mede a probabilidade atribuída à classe certa, não o `argmax` — já esteja melhorando.

É a escada da [seção 16.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/06-outras-funcoes-de-ativacao.html) de novo, invertida: lá a perda ficava presa em patamares enquanto o acerto não se mexia; aqui a perda desce reto e é o acerto que afunda antes de subir. Nos dois casos, a métrica que interessa passa um bom tempo sem mostrar o progresso que está acontecendo.

> **🔷 Conceito — A comparação, em números**
>
> | perda | epochs | tempo | acerto no treino | acerto no teste |
> |---|---|---|---|---|
> | `SSE` com sigmoide final ([16.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/06-outras-funcoes-de-ativacao.html)) | 300 | ~38 s | 0,884 | 0,890 |
> | `SoftmaxCrossEntropy` | 100 | ~13 s | 1,000 | 0,970 |
>
> Um terço dos epochs, um terço do tempo, e um resultado melhor nos dois conjuntos — com **a mesma arquitetura, a mesma inicialização, a mesma semente e o mesmo otimizador**. A única diferença é a última camada e a função de perda.
>
> Vale a ressalva de sempre: são 100 números de teste, e a diferença entre 0,890 e 0,970 são oito acertos. Isso sozinho caberia dentro do ruído. O que **não** cabe no ruído é a coluna do treino, com 923 exemplos: 0,884 contra 1,000. A rede com softmax aprendeu o conjunto de treino inteiro em 100 epochs; a outra não tinha chegado lá em 300.

### Os três erros

Agora que a saída é uma distribuição, dá para olhar os erros de um jeito que o capítulo anterior não permitia:

In [ ]:
rotulos = ["o próprio número", "fizz", "buzz", "fizzbuzz"]

for n in range(1, 101):
    previsto = argmax(net.forward(binary_encode(n)))
    correto = argmax(fizz_buzz_encode(n))
    if previsto != correto:
        bruta = net.forward(binary_encode(n))
        probs = softmax(bruta)
        print(f"n = {n:3d}  previu '{rotulos[previsto]}', era '{rotulos[correto]}'")
        print(f"         saída bruta {[round(v, 3) for v in bruta]}")
        print(f"         probabilidades {[round(p, 4) for p in probs]}"
              f"  soma {sum(probs):.4f}")

Três erros em cem, e eles são de três tipos diferentes — mas agora essa frase tem apoio numérico em vez de intuição. Para $n = 40$, as duas probabilidades disputam de perto: a rede está **genuinamente indecisa** entre "fizz" e "buzz", e a soma continua sendo 1, então essa disputa se lê como uma disputa de probabilidade. Para $n = 34$, a rede põe mais de 99% numa resposta errada: está **confiantemente errada**, que é sempre o pior tipo de erro. E $n = 4$ é o caso intermediário, o mais interessante dos três: 0,892 em "buzz", que é errado, mas com 0,105 sobrando na resposta certa. A rede está confiante e mesmo assim manteve a segunda hipótese viva — um erro que um limiar de confiança ("só aceite acima de 0,95") pegaria, enquanto o de $n = 34$ passaria batido.

A [seção 15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) fez exatamente essa distinção e teve de fazê-la de olho, comparando magnitudes de números que não somavam nada em particular. Aqui a distinção é uma leitura direta: 0,61 contra 0,39 é indecisão; 0,99 contra 0,01 é confiança. **A softmax não deixou o modelo mais certo — deixou o modelo mais legível.**

### Dropout

Como quase todo modelo de aprendizado de máquina, redes neurais tendem a **sobreajustar** os dados de treino. O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) definiu o problema, e a [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html) apresentou uma solução: penalizar coeficientes grandes, encolhendo os parâmetros em direção a zero.

O dropout é outra solução, e ela é estranha na primeira leitura. **Durante o treino, desligamos cada neurônio ao acaso** — substituímos a saída dele por 0 — com uma probabilidade fixa. A rede fica impedida de depender de qualquer neurônio individual, o que parece ajudar contra o sobreajuste.

Cabe aqui uma franqueza: o dropout entra neste livro **sem nenhuma medição de que ele serve**. Não é caso único — a [seção 13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) apresenta a máquina de vetores de suporte e diz, com todas as letras, que não a implementa — e, sem implementação, não há o que medir. A diferença é que lá a técnica fica de fora do código, e aqui ela entra na rede que treina de verdade. Com três passadas em 10.000 imagens, a rede da [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) fecha com 0,9020 de acerto no treino contra 0,8815 no teste: ela ainda não sobreajusta nada, e portanto o dropout ali não tem o que consertar — e não há, em página nenhuma, uma comparação com e sem. Ele entra porque o livro-texto o põe na rede, e porque o que interessa medir é o **comportamento** dele — a saída que muda entre treino e avaliação, e o prejuízo de esquecer de desligá-lo —, não o benefício, que exigiria um orçamento de treino que este livro não tem.

Na hora de avaliar, não queremos desligar neurônio nenhum. Então a camada precisa saber se está treinando ou não. E, como no treino ela deixa passar só uma fração da entrada, para que a saída seja comparável na avaliação ela **encolhe** as saídas uniformemente por essa mesma fração:

In [ ]:
class Dropout(Layer):
    def __init__(self, p: float) -> None:
        self.p = p
        self.train = True

    def forward(self, input: Tensor) -> Tensor:
        if self.train:
            # Cria uma máscara de 0s e 1s com a forma da entrada,
            # usando a probabilidade especificada.
            self.mask = tensor_apply(
                lambda _: 0 if random.random() < self.p else 1,
                input)
            # Multiplica pela máscara para desligar entradas.
            return tensor_combine(operator.mul, input, self.mask)
        else:
            # Na avaliação, só encolhe as saídas uniformemente.
            return tensor_apply(lambda x: x * (1 - self.p), input)

    def backward(self, gradient: Tensor) -> Tensor:
        if self.train:
            # Só propaga os gradientes onde a máscara vale 1.
            return tensor_combine(operator.mul, gradient, self.mask)
        else:
            raise RuntimeError("não chame backward fora do modo de treino")

Os dois modos, com um `p` exagerado para ficar visível:

In [ ]:
random.seed(7)

camada = Dropout(0.5)
entrada = [1.0] * 10

print("treino     :", camada.forward(entrada))
camada.train = False
print("avaliação  :", camada.forward(entrada))

No treino, metade dos valores foi zerada e a outra metade passou intacta — e o sorteio é diferente a cada chamada. Na avaliação, **nada** é zerado e tudo é multiplicado por $1 - p = 0{,}5$. As duas saídas têm a mesma soma esperada, que é o ponto do encolhimento: a camada seguinte recebe entradas da mesma magnitude típica nos dois modos, e por isso os pesos aprendidos no treino continuam valendo na avaliação.

> **⚠️ Atenção — O primeiro modelo deste livro que se comporta diferente conforme o momento**
>
> Até aqui, um modelo treinado era uma função: mesma entrada, mesma saída, sempre. Com dropout, isso deixa de valer, e de duas formas ao mesmo tempo.
>
> **A saída passa a ser aleatória durante o treino.** Duas chamadas de `forward` com a mesma entrada devolvem coisas diferentes, porque a máscara é sorteada de novo. É por isso que todo chunk desta seção leva `random.seed` — sem semente, o texto pararia de bater com a saída a cada renderização.
>
> **A saída depende de um atributo mutável, `self.train`.** Esquecer de pôr `train = False` antes de avaliar é o bug clássico do dropout, e ele não levanta erro nenhum: o modelo simplesmente avalia com neurônios desligados ao acaso, e a métrica sai pior e instável. Ninguém desconfia da métrica quando ela sai *pior* — desconfia-se do modelo. A [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) mede esse prejuízo num caso concreto.
>
> Repare que `backward` **levanta** `RuntimeError` fora do modo de treino — a única verificação defensiva no caminho de treino e avaliação de uma camada. (As outras duas da biblioteca guardam as bordas: o `ValueError` de inicialização desconhecida na [seção 16.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/03-a-camada-linear.html) e o `assert` de formas ao carregar pesos, na [16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html).) Ela protege o caso simétrico e mais raro; o caso comum, o de avaliar em modo de treino, não é protegido por nada.

> **💡 Dica — Na prática: `CrossEntropyLoss`, `model.train()` e `model.eval()`**
>
> **A perda.** No PyTorch, `nn.CrossEntropyLoss()` faz exatamente o que a nossa `SoftmaxCrossEntropy` faz: recebe as saídas **cruas** da rede — os *logits* — e aplica a softmax por dentro. A documentação avisa isso explicitamente, com a frase "the input is expected to contain the unnormalized logits for each class", justamente porque aplicar `nn.Softmax` na rede e depois usar essa perda é o erro comum descrito acima. No `scikit-learn`, o `MLPClassifier` faz o mesmo sem oferecer escolha: para mais de duas classes, ele usa softmax na saída e entropia cruzada como perda, o que é verificável em `out_activation_` e `loss` — e é exatamente o que a caixa final do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html) constatou.
>
> **Os modos.** O `self.train` da nossa camada existe lá como um método do modelo inteiro:
>
> ```python
> modelo.train()   # liga o dropout, e o que mais depender do modo
> saida = modelo(x)
>
> modelo.eval()    # desliga
> with torch.no_grad():
>     saida = modelo(x_teste)
> ```
>
> Chamar `.train()` ou `.eval()` no modelo propaga o modo para **todas** as camadas de dentro, recursivamente. É uma melhoria real sobre o nosso arranjo, em que cada `Dropout` precisa ser guardado numa variável para poder ser alternado à mão — como a [seção 16.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/08-exemplo-mnist.html) vai fazer, de forma desconfortável.
>
> **Uma diferença de convenção que vale saber.** O `nn.Dropout` do PyTorch **não** encolhe as saídas na avaliação, como o nosso faz. Ele faz o contrário: durante o treino, além de zerar, ele **amplia** os valores sobreviventes por $1/(1-p)$, e na avaliação não faz nada. Isso se chama *inverted dropout*, e o efeito esperado é o mesmo — a diferença é que a avaliação fica sendo a identidade, o que é mais barato e permite exportar o modelo sem carregar o `p` junto. Se você comparar as duas implementações valor a valor, elas não coincidem; se comparar as médias, coincidem.

## Exemplo: MNIST

> **📌 Nota**
>
> Esta seção corresponde a *Example: MNIST* e a *Saving and Loading Models*, do capítulo 19 de Grus (2019).

O MNIST é um conjunto de dígitos manuscritos que todo mundo usa para aprender deep learning. São 70.000 imagens de 28 × 28 pixels em tons de cinza, cada uma com o dígito que ela representa: 60.000 para treino e 10.000 para teste.

É a última coisa que este capítulo constrói, e a mais completa: ela usa **tudo**: os tensores, as camadas, a rede como sequência, a perda de entropia cruzada, o otimizador com momento e o dropout. O laço de treino é o mesmo desde a seção do XOR.

### O arquivo binário, aberto à mão

O livro-texto resolve a leitura dos dados instalando um pacote:

```python
python -m pip install mnist
```

```python
import mnist
mnist.temporary_dir = lambda: '/tmp'
train_images = mnist.train_images().tolist()
```

> **❗ Importante — Por que esse `import` não existe aqui**
>
> Aquele `mnist.train_images()` **baixa** os dados da internet. Isso está proibido duas vezes neste livro.
>
> A primeira razão é de escopo: o pacote `mnist` só serve para baixar um arquivo que vamos ler do disco de qualquer jeito, e por isso ele ficou de fora do ambiente desta disciplina, junto com o pacote de acesso ao Twitter que o livro-texto usa em outro capítulo.
>
> A segunda é mais dura: **nenhum byte deste livro vem da rede em tempo de renderização**. É uma invariante verificada — o livro é renderizado num contêiner com a rede desligada, e um `requests.get` ou um download escondido faria a construção falhar. O motivo não é purismo: uma página que baixa dados na hora de renderizar produz um livro que muda sozinho quando a fonte muda, e a mudança é silenciosa.
>
> Então os quatro arquivos do MNIST, no formato original, estão versionados em `dados/mnist/`. E lê-los é, por si só, uma caixa-preta a menos: o `mnist.train_images()` do livro-texto faz exatamente o que as duas funções abaixo fazem, mais um download.

O formato se chama **IDX** e é simples o bastante para caber em seis linhas. Um arquivo IDX começa com um *número mágico* de quatro bytes que identifica o tipo de conteúdo, seguido de um inteiro de quatro bytes por dimensão, e depois os dados crus, um byte por pixel. Tudo em *big-endian*, que é o `>` do formato de `struct`:

In [ ]:
import gzip
import struct
from typing import List
from scratch.deep_learning import Tensor

def ler_imagens_idx(caminho: str) -> List[Tensor]:
    with gzip.open(caminho, "rb") as f:
        magico, n, linhas, colunas = struct.unpack(">IIII", f.read(16))
        assert magico == 2051                    # 2051 = arquivo de imagens
        buf = f.read(n * linhas * colunas)
    tam = linhas * colunas
    return [list(buf[i * tam:(i + 1) * tam]) for i in range(n)]

def ler_rotulos_idx(caminho: str) -> List[int]:
    with gzip.open(caminho, "rb") as f:
        magico, n = struct.unpack(">II", f.read(8))
        assert magico == 2049                    # 2049 = arquivo de rótulos
        return list(f.read(n))

Os dois números mágicos, 2051 e 2049, são a única parte que precisa ser consultada na especificação; o resto se lê do próprio código. Repare que as imagens já saem **achatadas**: cada uma vira uma lista de 784 inteiros, e não uma lista de 28 listas de 28. O livro-texto achata num passo separado, mais adiante; aqui o leitor já entrega assim, porque é dessa forma que a camada linear precisa dos dados.

In [ ]:
import time
from scratch.deep_learning import shape

t0 = time.time()

imagens_treino = ler_imagens_idx("dados/mnist/train-images-idx3-ubyte.gz")
rotulos_treino = ler_rotulos_idx("dados/mnist/train-labels-idx1-ubyte.gz")
imagens_teste = ler_imagens_idx("dados/mnist/t10k-images-idx3-ubyte.gz")
rotulos_teste = ler_rotulos_idx("dados/mnist/t10k-labels-idx1-ubyte.gz")

print(f"leitura em {time.time() - t0:.2f} s")
print("treino:", shape(imagens_treino), " rótulos:", shape(rotulos_treino))
print("teste :", shape(imagens_teste), " rótulos:", shape(rotulos_teste))
print("dez primeiros rótulos:", rotulos_treino[:10])

Setenta mil imagens em menos de um segundo, sem `numpy` e sem rede. Os dez primeiros rótulos são `[5, 0, 4, 1, 9, 2, 1, 3, 1, 4]`, que são os canônicos do MNIST — se a leitura estivesse errada, essa sequência não bateria.

E dá para conferir com os olhos, que é sempre melhor. As imagens estão achatadas, então precisamos dobrá-las de volta para desenhar:

In [ ]:
# Figura: As cem primeiras imagens do conjunto de treino do MNIST
from matplotlib import pyplot as plt

def como_matriz(imagem: Tensor) -> Tensor:
    return [imagem[linha * 28:(linha + 1) * 28] for linha in range(28)]

fig, ax = plt.subplots(10, 10, figsize=(7, 7))

for i in range(10):
    for j in range(10):
        ax[i][j].imshow(como_matriz(imagens_treino[10 * i + j]), cmap='Greys')
        ax[i][j].xaxis.set_visible(False)
        ax[i][j].yaxis.set_visible(False)

plt.tight_layout()
plt.show()

São de fato dígitos manuscritos, na orientação certa e não espelhados.

> **📌 Nota**
>
> O `cmap='Greys'` da chamada acima é o livro-texto sendo honesto sobre o próprio processo: a primeira tentativa dele produziu números amarelos sobre fundo preto, e ele diz não ser nem esperto nem sutil o bastante para saber que precisava daquele argumento — procurou no Google e achou a resposta. É um bom retrato do trabalho de verdade: uma parte dele é saber o que se quer, e outra é descobrir o nome da opção que faz aquilo.

### Preparando os dados

Duas transformações. A primeira é dividir por 256, para trazer os valores de $[0, 255]$ para perto de $[0, 1]$. A segunda é **centralizar**: uma rede treina melhor quando as entradas têm média zero, então subtraímos o pixel médio antes de dividir.

In [ ]:
from scratch.deep_learning import tensor_sum

media = tensor_sum(imagens_treino) / 60000 / 784
print(f"pixel médio do conjunto de treino: {media:.4f}")

Trinta e três, numa escala de 0 a 255. Faz sentido: a maior parte de uma imagem do MNIST é fundo preto.

> **❗ Importante — O orçamento de renderização, que é uma decisão de conteúdo**
>
> Aqui é onde este capítulo cobra o preço de tudo o que fez.
>
> A rede que vamos treinar leva, **medido nesta máquina**, cerca de 6 milissegundos por imagem. Uma passada completa pelas 60.000 imagens de treino custa, portanto, algo em torno de **seis minutos**. O livro-texto avisa o mesmo em outras palavras: no laptop dele, o treino levava mais de vinte minutos.
>
> Só que este livro é gerado por uma máquina, do zero, a cada publicação — e o cache de execução é apagado de propósito antes disso, justamente para provar que nenhuma página depende da rede. Não há como esconder seis minutos atrás de um cache que é apagado.
>
> Então o corte é este, e ele está escrito aqui em vez de disfarçado:
>
> | | livro-texto | aqui |
> |---|---|---|
> | imagens de treino | 60.000 | **10.000** |
> | imagens de teste | 10.000 | **2.000** |
> | passadas pelo treino | 1 | **3** |
>
> Três passadas, e não duas, por um motivo que a tabela mais adiante deixa ver: **com duas passadas a rede profunda ainda perdia para a regressão logística** — 0,8595 contra 0,8620 no teste —, e a seção terminaria sugerindo que sete camadas não compram nada. A terceira passada custa mais um minuto de renderização e inverte o resultado. É o orçamento decidindo o que o capítulo consegue afirmar, que é exatamente a lição desta caixa.
>
> Isso é o assunto, não um contratempo. **A lentidão é o preço da transparência**, e este é, com folga, o capítulo mais caro do livro. Outras páginas também cronometram os chunks pesados — seis segundos no [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html), noventa no [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) —, mas este é o único em que o preço **decide o que o capítulo consegue afirmar**, e não apenas quanto ele demora para renderizar. Uma biblioteca vetorizada rodando numa GPU faz esta mesma rede, sobre o conjunto inteiro, em segundos — e é exatamente por isso que GPUs existem. Um aluno que só chamou `.fit()` nunca teve como sentir a diferença entre "seis minutos" e "seis segundos" como consequência de uma escolha de representação.
>
> Fica como exercício o que o corte custou: rodar com as 60.000 imagens e mais passadas, e comparar com os 92% que o livro-texto relata para a mesma rede.

In [ ]:
from scratch.deep_learning import one_hot_encode

N_TREINO, N_TESTE = 10000, 2000

X_treino = [[(pixel - media) / 256 for pixel in imagem]
            for imagem in imagens_treino[:N_TREINO]]
X_teste = [[(pixel - media) / 256 for pixel in imagem]
           for imagem in imagens_teste[:N_TESTE]]

y_treino = [one_hot_encode(r) for r in rotulos_treino[:N_TREINO]]
y_teste = [one_hot_encode(r) for r in rotulos_teste[:N_TESTE]]

print("X:", shape(X_treino), shape(X_teste))
print("y:", shape(y_treino), shape(y_teste))
print("média do pixel depois de centralizar:",
      round(tensor_sum(X_treino) / N_TREINO / 784, 6))

O `one_hot_encode` transforma um rótulo num vetor de dez posições, com um 1 na posição do dígito. É a mesma ideia do `fizz_buzz_encode` do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-exemplo-fizz-buzz.html), generalizada:

In [ ]:
assert one_hot_encode(3) == [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
assert one_hot_encode(2, num_labels=5) == [0, 0, 1, 0, 0]

one_hot_encode(rotulos_treino[0]), rotulos_treino[0]

### O laço, uma vez só

Uma das forças das abstrações deste capítulo é que o **mesmo** laço serve para modelos diferentes. Vamos escrevê-lo antes de ter um modelo: ele recebe o modelo, os dados, a perda e — se for para treinar — um otimizador. Sem otimizador, ele apenas avalia.

In [ ]:
import tqdm
from scratch.deep_learning import Layer, Loss, Optimizer
from scratch.neural_networks import argmax

def loop(model: Layer,
         images: List[Tensor],
         labels: List[Tensor],
         loss: Loss,
         optimizer: Optimizer = None) -> tuple:
    correct = 0         # conta as previsões certas
    total_loss = 0.0    # acumula a perda

    with tqdm.trange(len(images)) as t:
        for i in t:
            predicted = model.forward(images[i])            # prevê
            if argmax(predicted) == argmax(labels[i]):      # confere
                correct += 1
            total_loss += loss.loss(predicted, labels[i])   # calcula a perda

            # Se estamos treinando, retropropaga o gradiente e atualiza os pesos.
            if optimizer is not None:
                gradient = loss.gradient(predicted, labels[i])
                model.backward(gradient)
                optimizer.step(model)

            # E atualiza as métricas na barra de progresso.
            avg_loss = total_loss / (i + 1)
            acc = correct / (i + 1)
            t.set_description(f"mnist loss: {avg_loss:.3f} acc: {acc:.3f}")

    return correct / len(images), total_loss / len(images)

> **📌 Nota**
>
> Duas observações sobre este laço.
>
> A primeira: o `tqdm` desenha uma barra de progresso que é útil no terminal e ilegível numa página, então o chunk leva `#| warning: false`. Por isso o laço também **devolve** a acurácia e a perda média, coisa que a versão do livro-texto não faz — lá, os números só aparecem na barra.
>
> A segunda, mais importante: **a acurácia que este laço mede durante o treino é uma acurácia corrente.** Ela conta os acertos ao longo da passada, incluindo os das primeiras centenas de imagens, quando o modelo ainda mal começou a aprender. Não é a acurácia do modelo no fim da passada; é a média do desempenho dele **enquanto** aprendia. Isso torna o número do primeiro epoch sistematicamente pessimista, e vai ficar visível daqui a pouco.

### Uma linha de base: regressão logística

Antes de qualquer rede profunda, vale medir o mais simples que a biblioteca permite. Uma **regressão logística multiclasse** é uma camada linear seguida de softmax — e, como a softmax mora na perda, isso é literalmente `Linear(784, 10)`.

O modelo procura dez funções lineares tais que, se a entrada representa um 5, a quinta função produz a maior saída. É o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) com dez classes em vez de duas, como a [seção 16.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/07-softmax-e-dropout.html) mostrou.

In [ ]:
import random
from scratch.deep_learning import Linear, SoftmaxCrossEntropy, Momentum

random.seed(0)

modelo_linear = Linear(784, 10)
loss = SoftmaxCrossEntropy()
optimizer = Momentum(learning_rate=0.01, momentum=0.99)

t0 = time.time()
acc_treino, perda_treino = loop(modelo_linear, X_treino, y_treino, loss, optimizer)
segundos = time.time() - t0

acc_teste, perda_teste = loop(modelo_linear, X_teste, y_teste, loss)

print(f"treino: acurácia corrente {acc_treino:.4f}, perda média {perda_treino:.4f}")
print(f"teste : acurácia {acc_teste:.4f}, perda média {perda_teste:.4f}")
print(f"{segundos:.1f} s para uma passada, {1000 * segundos / N_TREINO:.2f} ms por imagem")

Uma passada por 10.000 imagens, 7.850 parâmetros, e o modelo já acerta cerca de 86% do conjunto de teste. O livro-texto, com as 60.000 imagens, relata cerca de 89% — a diferença é o corte de escopo desta página, medida.

Repare no custo por imagem: cerca de 2 milissegundos, para um modelo de uma camada só. Multiplique pelas 60.000 imagens do conjunto completo e são mais de dois minutos, para o modelo **mais barato** que este capítulo consegue construir.

### A rede profunda

Agora a rede de verdade: duas camadas escondidas, a primeira com 30 neurônios e a segunda com 10, ativação `Tanh`, e dropout entre a camada linear e a ativação.

In [ ]:
from scratch.deep_learning import Sequential, Tanh, Dropout

random.seed(0)

# guardamos as camadas de dropout em variáveis para poder ligar e desligar o treino
dropout1 = Dropout(0.1)
dropout2 = Dropout(0.1)

modelo = Sequential([
    Linear(784, 30),   # camada escondida 1: tamanho 30
    dropout1,
    Tanh(),
    Linear(30, 10),    # camada escondida 2: tamanho 10
    dropout2,
    Tanh(),
    Linear(10, 10)     # camada de saída: tamanho 10
])

print("parâmetros por tensor:", [shape(p) for p in modelo.params()])
print("total de parâmetros:",
      sum(len(p) * (len(p[0]) if isinstance(p[0], list) else 1)
          for p in modelo.params()))

> **📌 Nota — O `Dropout` está **antes** da ativação, e isso é incomum**
>
> Repare na ordem: `Linear`, `Dropout`, `Tanh`. A posição usual é a outra — desligar neurônios depois da ativação —, e quem leu a [seção 16.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/07-softmax-e-dropout.html) vai estranhar. Aqui é o que o livro-texto escreve, e mantivemos.
>
> Com a `Tanh`, a diferença é pequena, e o motivo é que $\tanh(0) = 0$: zerar a pré-ativação e depois aplicar a `tanh` dá zero, e aplicar a `tanh` e depois zerar também dá zero. Com uma ativação que não passa pela origem — a sigmoide, em que $\sigma(0) = 1/2$ — as duas ordens dariam coisas diferentes, e esta seria a errada.

Sete camadas, quase 24 mil parâmetros — e o laço de treino continua sendo o mesmo. A única complicação é ligar o dropout para treinar e desligá-lo para avaliar:

In [ ]:
optimizer = Momentum(learning_rate=0.01, momentum=0.99)

historico = []

for epoch in range(3):
    dropout1.train = dropout2.train = True          # liga o dropout e treina
    t0 = time.time()
    acc_treino, perda_treino = loop(modelo, X_treino, y_treino, loss, optimizer)
    segundos = time.time() - t0

    dropout1.train = dropout2.train = False         # desliga e avalia
    acc_teste, perda_teste = loop(modelo, X_teste, y_teste, loss)

    historico.append((epoch + 1, acc_treino, acc_teste, segundos))

In [ ]:
print("epoch   treino (corrente)   teste   segundos   ms/imagem")
for epoch, acc_tr, acc_te, seg in historico:
    print(f"{epoch:5d}   {acc_tr:16.4f}   {acc_te:.4f}   {seg:8.1f}"
          f"   {1000 * seg / N_TREINO:9.2f}")

Três leituras dessa tabela.

**O primeiro epoch parece um desastre e não é.** A acurácia de treino do primeiro epoch é bem menor que a de teste do mesmo epoch — o que seria impossível se as duas medissem a mesma coisa. Elas não medem: a de treino é a acurácia **corrente**, que inclui as primeiras centenas de imagens, quando o modelo era pesos aleatórios. A de teste é medida depois, com o modelo já treinado por uma passada inteira.

**A rede profunda supera a regressão logística — por pouco.** Ao fim do terceiro epoch ela passa de 88% no teste, contra os cerca de 86% do modelo linear. Vale dizer com franqueza: são 2.000 imagens de teste, e uma diferença de dois pontos percentuais está no limite do que essa amostra distingue com segurança. O que a tabela mostra sem ambiguidade é a **tendência** — a rede profunda ainda estava melhorando de epoch para epoch quando o orçamento acabou, e o livro-texto, com o conjunto inteiro, relata mais de 92% para ela contra 89% do modelo linear.

**O custo por imagem quase triplicou.** Cerca de 6 milissegundos contra pouco mais de 2 do modelo linear, o que dá cerca de 2,7 vezes — e a rede profunda tem cerca de três vezes mais parâmetros. É aproximadamente proporcional, e é o que se espera: quase todo o tempo é gasto nas multiplicações da camada de 784 × 30, que sozinha responde por 23.520 dos 23.970 parâmetros.

### O bug do dropout, medido

A [seção 16.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/07-softmax-e-dropout.html) avisou que esquecer de desligar o dropout na avaliação é o erro clássico. Vale ver o tamanho do estrago:

In [ ]:
random.seed(0)

dropout1.train = dropout2.train = True              # esquecemos de desligar
acc_com, _ = loop(modelo, X_teste, y_teste, loss)

dropout1.train = dropout2.train = False             # como deveria ser
acc_sem, _ = loop(modelo, X_teste, y_teste, loss)

print(f"avaliando com dropout ligado:   {acc_com:.4f}")
print(f"avaliando com dropout desligado:{acc_sem:.4f}")

Quase três pontos percentuais de acurácia jogados fora, sem erro, sem aviso, e com o resultado variando a cada execução porque a máscara é sorteada de novo — só há um número estável aqui porque o chunk fixa a semente. **O modelo está certo; a medição é que está errada.** É um erro particularmente traiçoeiro porque a métrica sai *pior*, e uma métrica pior faz suspeitar do modelo, não do código que a mediu.

### Onde a rede erra

Os erros não se distribuem por igual entre os dígitos:

In [ ]:
from collections import Counter

confusoes = Counter()
for i in range(N_TESTE):
    previsto = argmax(modelo.forward(X_teste[i]))
    correto = argmax(y_teste[i])
    if previsto != correto:
        confusoes[(correto, previsto)] += 1

print("as cinco confusões mais comuns:")
for (correto, previsto), quantas in confusoes.most_common(5):
    print(f"  {correto} previsto como {previsto}: {quantas} vezes")

Os pares que aparecem no topo não são aleatórios: são os dígitos que **se parecem escritos à mão**. Um 4 mal fechado vira um 9; um 7 com a haste curva vira um 9 ou um 2; um 5 apressado vira um 3 ou um 8. A rede não sabe nada sobre a forma dos algarismos — ela recebeu 784 números soltos, sem noção de vizinhança entre pixels —, e ainda assim os erros dela caem exatamente onde cairiam os de um humano com pressa.

> **📌 Nota — O que falta, e o nome disso**
>
> Aquele "sem noção de vizinhança entre pixels" é a limitação central deste modelo. Para a nossa `Linear(784, 30)`, os pixels 100 e 101 — vizinhos na imagem — são tão relacionados quanto os pixels 3 e 700. Embaralhar as 784 colunas de todas as imagens da mesma forma **não mudaria nada no que o modelo pode aprender**. Os números não sairiam idênticos — os pesos iniciais são sorteados numa ordem fixa e não acompanhariam a permutação, então a inicialização efetiva seria outra —, mas seriam estatisticamente equivalentes, e o resultado, igualmente bom. Para este modelo, a posição de um pixel na imagem simplesmente não é informação.
>
> Os modelos que dominam o MNIST usam **camadas convolucionais**, que aplicam o mesmo pequeno conjunto de pesos deslizando sobre a imagem, e portanto sabem que pixels vizinhos são vizinhos. Elas se encaixariam na `Layer` desta biblioteca sem alteração nenhuma na interface — só o `forward` e o `backward` seriam outros. O que impede não é a arquitetura, é o custo: uma convolução sobre listas aninhadas em Python puro seria muitas vezes mais lenta do que o que esta página já paga.
>
> O site do MNIST descreve uma variedade de modelos que superam largamente os nossos. Muitos deles poderiam ser implementados com as peças construídas aqui, e nenhum deles terminaria de treinar.

### Salvando e carregando modelos

Estes modelos demoram para treinar, então seria bom poder guardá-los. Com o módulo `json` da biblioteca padrão, isso é curto: `params()` já devolve a lista de tensores de pesos, e um tensor é uma lista aninhada, que é exatamente o que o JSON serializa.

In [ ]:
import json

def save_weights(model: Layer, filename: str) -> None:
    weights = list(model.params())
    with open(filename, 'w') as f:
        json.dump(weights, f)

def load_weights(model: Layer, filename: str) -> None:
    with open(filename) as f:
        weights = json.load(f)

    # Confere a consistência
    assert all(shape(param) == shape(weight)
               for param, weight in zip(model.params(), weights))

    # E carrega usando atribuição de fatia:
    for param, weight in zip(model.params(), weights):
        param[:] = weight

O `assert` no meio de `load_weights` é a única salvaguarda: ele impede carregar os pesos de uma rede profunda dentro de uma rede rasa, ou coisa parecida. Note que só as **formas** são conferidas — nada guarda a arquitetura em si, então é você quem precisa instanciar o modelo certo antes de carregar.

E a `load_weights` termina com `param[:] = weight`, a atribuição de fatia da [seção 16.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/05-perda-e-otimizacao.html). Pelo mesmo motivo de lá: `param = weight` trocaria o nome local e deixaria o modelo intacto.

In [ ]:
save_weights(modelo, "/tmp/pesos-mnist.json")

import os
print(f"tamanho do arquivo: {os.path.getsize('/tmp/pesos-mnist.json') / 1024:.0f} KB")

# uma rede nova, com a mesma arquitetura e pesos aleatórios
random.seed(1)
d1, d2 = Dropout(0.1), Dropout(0.1)
copia = Sequential([Linear(784, 30), d1, Tanh(),
                    Linear(30, 10), d2, Tanh(),
                    Linear(10, 10)])
d1.train = d2.train = False

antes, _ = loop(copia, X_teste, y_teste, loss)
load_weights(copia, "/tmp/pesos-mnist.json")
depois, _ = loop(copia, X_teste, y_teste, loss)

print(f"acurácia da cópia antes de carregar:  {antes:.4f}")
print(f"acurácia da cópia depois de carregar: {depois:.4f}")

Antes de carregar, a cópia acerta cerca de um décimo — que é o que se espera de dez classes e pesos aleatórios. Depois, ela acerta exatamente o mesmo que o modelo original: os pesos são os mesmos números.

> **📌 Nota**
>
> O JSON guarda os dados como **texto**, o que é uma representação extremamente ineficiente — cada peso vira uma sequência de caracteres com o sinal, o ponto e dezenas de dígitos. Numa aplicação de verdade você usaria a biblioteca `pickle`, que serializa para um formato binário mais compacto; o livro-texto escolheu manter simples e legível para humanos, e nós mantemos a escolha dele.
>
> O arquivo grava em `/tmp` de propósito: ele é um subproduto da renderização, e não faz parte do repositório deste livro. Num trabalho de verdade, o lugar dele seria junto do código que treinou o modelo — e junto de alguma anotação sobre qual arquitetura instanciar antes de carregá-lo, já que o arquivo não guarda essa informação.

> **🔷 Conceito — O fim do arco**
>
> O [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) construiu uma rede que resolvia um problema de brinquedo com dez bits de entrada, e o fez com uma retropropagação escrita à mão que só funcionava para duas camadas.
>
> Nesta página, sete camadas e vinte e quatro mil parâmetros reconhecem dígitos manuscritos, e **nenhuma linha de gradiente foi escrita para isso**. O laço de treino é o mesmo do XOR. Trocar `Tanh` por `Relu`, `SoftmaxCrossEntropy` por `SSE`, `Momentum` por `GradientDescent`, ou acrescentar mais uma camada escondida — cada uma dessas mudanças é uma edição de uma linha, e nenhuma delas toca no resto do código.
>
> É isso que uma abstração compra, e é isso que está do outro lado quando alguém escreve `Sequential([Linear(784, 30), Tanh(), Linear(30, 10)])` em qualquer framework. Não é mágica, não é um algoritmo escondido: são as oito seções deste capítulo, escritas em C e em CUDA em vez de em listas do Python.
>
> E a diferença entre as duas versões — a que você leu e a que roda em produção — é de velocidade, não de ideia. Esta página levou minutos para fazer o que uma GPU faz em segundos. **Você agora sabe exatamente o que aqueles segundos estão fazendo.**
>
> Falta um capítulo, e repare no que esta seção precisou para existir: a tabela de confusões acima só pôde ser montada porque, para cada uma das duas mil imagens de teste, alguém já tinha escrito qual dígito era. O [Capítulo 17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html) trabalha sem essa coluna — e é a última coisa que este livro constrói.

> **💡 Dica — Na prática: o mesmo modelo, em PyTorch**
>
> O modelo desta página, escrito num framework:
>
> ```python
> import torch
> import torch.nn as nn
>
> modelo = nn.Sequential(
>     nn.Linear(784, 30), nn.Dropout(0.1), nn.Tanh(),
>     nn.Linear(30, 10),  nn.Dropout(0.1), nn.Tanh(),
>     nn.Linear(10, 10),
> )
>
> perda = nn.CrossEntropyLoss()
> otimizador = torch.optim.SGD(modelo.parameters(), lr=0.0001, momentum=0.99)
>
> for epoch in range(3):
>     modelo.train()
>     for X, y in carregador_de_treino:      # lotes de 64 imagens de uma vez
>         otimizador.zero_grad()
>         perda(modelo(X), y).backward()
>         otimizador.step()
> ```
>
> A arquitetura é a mesma, linha por linha. **A taxa de aprendizado não é**, e o desconto de cem vezes não é engano de digitação: é a conta da [seção 16.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/05-perda-e-otimizacao.html) aplicada a este $\mu$. O nosso `Momentum` acumula uma média móvel do gradiente, cujo limite é o próprio $g$; o `optim.SGD`, com o padrão `dampening=0`, acumula uma soma cujo limite é $g/(1-\mu)$. Com o $\mu = 0{,}9$ daquela seção isso valia dez vezes; com o $\mu = 0{,}99$ desta, vale **cem**. Traduzir `Momentum(learning_rate=0.01, momentum=0.99)` como `lr=0.01` daria um passo cem vezes maior que o nosso — e quanto mais perto de 1 fica o momento, mais cara sai a distração.
>
> Três diferenças no que está em volta dela:
>
> **Lotes.** O `carregador_de_treino` entrega 64 imagens de uma vez, e `modelo(X)` processa as 64 numa multiplicação de matriz só. O nosso laço faz uma imagem por vez, porque a nossa `Linear` não sabe fazer outra coisa. Sozinho, isso costuma valer uma ordem de grandeza.
>
> **Modo de treino propagado.** `modelo.train()` e `modelo.eval()` alcançam todas as camadas de dentro. As nossas duas variáveis `dropout1` e `dropout2`, ligadas e desligadas à mão, existem porque não temos esse mecanismo — e o desconforto de escrevê-las é a melhor propaganda que ele podia ter.
>
> **Persistência.** `torch.save(modelo.state_dict(), caminho)` guarda um dicionário de tensores binários, com os nomes das camadas junto. O nosso JSON de texto guarda uma lista sem nomes, e é por isso que o `load_weights` só consegue conferir formas.
>
> Sobre o tempo: a rede desta página levou cerca de um minuto por passada em 10.000 imagens. A mesma rede em PyTorch, em CPU e com lotes, faz as 60.000 em poucos segundos; em GPU, mais rápido ainda. **Nada disso muda a conta que está sendo feita** — continuam sendo produtos escalares, derivadas de `tanh` e passos de gradiente com momento, exatamente como nas oito seções anteriores.
>
> E se o assunto for reconhecer dígitos de verdade, ninguém escreveria nem uma coisa nem outra: `sklearn.datasets.load_digits` mais três linhas de `MLPClassifier` resolvem o problema em segundos, e o `scikit-learn` nem sequer expõe camadas para você compor. O que você ganhou nestas oito seções não foi um classificador de dígitos. Foi saber o que existe atrás daquelas três linhas.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 19 de Grus (2019) começa reconhecendo que o capítulo mal arranha a superfície do assunto, e que há muitos bons livros e posts sobre deep learning — e muitos, muitos ruins. Ela faz três indicações:

- *Deep Learning*, de Ian Goodfellow, Yoshua Bengio e Aaron Courville (MIT Press), é o livro-texto canônico da área e está [disponível de graça na internet](https://www.deeplearningbook.org/). O livro-texto o descreve como muito bom, com a ressalva de que envolve bastante matemática.
- *Deep Learning with Python*, de François Chollet (Manning), é uma introdução ao **Keras** — a biblioteca em que o desenho deste capítulo se inspira. Se a estrutura de `Layer`, `Sequential` e `Optimizer` pareceu familiar, é por isso.
- O próprio autor diz usar principalmente o [PyTorch](https://pytorch.org/), cujo site traz documentação e tutoriais.

Vale acrescentar uma leitura que não está na lista dele: G{\'e}ron (2022) cobre o mesmo terreno pelo lado prático, com Keras e TensorFlow, e é o ponto de partida usual para quem sai daqui e quer treinar redes de verdade.

## Referências

- **G{\'e}ron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed.. O'Reilly Media. 2022.
- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.